## Multimodal Ranking Model

In [1]:
import os
import math
import random
from collections import defaultdict
from typing import List, Dict, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import _LRScheduler
from torch.utils.data import Dataset, DataLoader, Sampler


# ================== 全局与超参 ==================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 数据列规格
DAILY_T, DAILY_F = 5, 43      # 5 步 × 43 维
MIN_T, MIN_F     = 5, 18      # 5 步 × 18 维
FEAT_TOTAL       = DAILY_T * DAILY_F + MIN_T * MIN_F

# embedding 目录
TELEGRAM_EMB_DIR = "/root/autodl-tmp/telegram_emb_cache_clean"

# 训练批次采样（按“天”为组）
DAY_FORWARD_CHUNK = 4096

BATCH_WORKERS    = 4
PIN_MEMORY       = True

# 训练超参
EPOCHS           = 12
LR               = 1e-4
MIN_LR           = 3e-6
WEIGHT_DECAY     = 1e-4
MAX_GRAD_NORM    = 5.0
USE_AMP          = False
WARMUP_EPOCHS    = 2

# 温度退火
TAU_START        = 2.0
TAU_END          = 0.9
TAU_ANNEAL_EPS   = 3

# 模型宽度
LSTM_HIDDEN      = 128
ATTN_HIDDEN      = 64
MIN_CONV_C       = 128
FUSE_DIM         = 128
DROPOUT          = 0.22

# 业务目标
N_TARGET         = 20
OPEN_THR         = 0       # 改为 0

# 熵正则
ALPHA_ENTROPY    = 0.0

# 评估
EVAL_TOP_LIST    = [5, 10, 20]

# ========= 新增：早停/选模所依据的 K =========
EARLY_STOP_K     = 5

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # 新版 PyTorch 可加：
    # torch.use_deterministic_algorithms(True)
    
seed_all()


# ================== 数据加载 ==================
TRAIN_CSV = "/root/autodl-tmp/train_filtered_filtered.csv"
VAL_CSV   = "/root/autodl-tmp/val_filtered_filtered.csv"
TEST_CSV  = "/root/autodl-tmp/test_filtered_filtered.csv"

train_df = pd.read_csv(TRAIN_CSV, low_memory=False)
val_df   = pd.read_csv(VAL_CSV,   low_memory=False)
test_df  = pd.read_csv(TEST_CSV,  low_memory=False)

# label clip（用于训练 loss）
p1  = train_df["opening_gains"].quantile(0.01)
p99 = train_df["opening_gains"].quantile(0.99)
train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
test_df["opening_gains_clipped"]  = test_df["opening_gains"].clip(lower=p1, upper=p99)

def cast_df(df: pd.DataFrame):
    feat_start = 2
    feat_end   = 2 + FEAT_TOTAL
    feats = df.iloc[:, feat_start:feat_end].astype(np.float32).values
    y_raw  = df["opening_gains"].astype(np.float32).values
    y_clip = df["opening_gains_clipped"].astype(np.float32).values
    code = df.iloc[:, 0].astype(str).values
    date = df.iloc[:, 1].astype(str).values
    return code, date, feats, y_clip, y_raw

tr_code, tr_date, tr_feats, tr_y, tr_y_raw = cast_df(train_df)
va_code, va_date, va_feats, va_y, va_y_raw = cast_df(val_df)
te_code, te_date, te_feats, te_y, te_y_raw = cast_df(test_df)

def build_groups(date_arr: np.ndarray) -> Dict[str, np.ndarray]:
    groups = defaultdict(list)
    for idx, d in enumerate(date_arr):
        groups[d].append(idx)
    return {k: np.array(v, dtype=np.int64) for k, v in groups.items()}

train_groups = build_groups(tr_date)
val_groups   = build_groups(va_date)
test_groups  = build_groups(te_date)

tr_dates = sorted(train_groups.keys())
va_dates = sorted(val_groups.keys())
te_dates = sorted(test_groups.keys())

tr_date2gid = {d: i for i, d in enumerate(tr_dates)}
va_date2gid = {d: i for i, d in enumerate(va_dates)}
te_date2gid = {d: i for i, d in enumerate(te_dates)}


# ================== embedding 读取 ==================
def load_embeddings(split: str, expected_n: int) -> np.ndarray:
    path = os.path.join(TELEGRAM_EMB_DIR, f"{split}_telegram_emb_filtered_filtered.npy")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Embedding file not found: {path}")
    emb = np.load(path).astype(np.float32)
    assert emb.ndim == 2, f"{split} emb must be 2D, got {emb.ndim}D"
    assert emb.shape[0] == expected_n, f"{split} emb rows mismatch: {emb.shape[0]} vs {expected_n}"
    print(f"[Info] Loaded {split} embeddings: shape={emb.shape} from {path}")
    return emb

tr_text = load_embeddings("train", tr_feats.shape[0])
va_text = load_embeddings("val",   va_feats.shape[0])
te_text = load_embeddings("test",  te_feats.shape[0])

TEXT_HIDDEN = tr_text.shape[1]
assert va_text.shape[1] == TEXT_HIDDEN and te_text.shape[1] == TEXT_HIDDEN, \
    f"text hidden mismatch across splits: train={TEXT_HIDDEN}, val={va_text.shape[1]}, test={te_text.shape[1]}"


# ================== Scheduler ==================
class WarmupCosine(_LRScheduler):
    def __init__(self, optimizer, warmup_epochs=1, max_epochs=EPOCHS, min_lr=1e-6, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.max_epochs = max_epochs
        self.min_lr = min_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        epoch = self.last_epoch + 1
        lrs = []
        for base_lr in self.base_lrs:
            if epoch <= self.warmup_epochs:
                lr = base_lr * epoch / max(1, self.warmup_epochs)
            else:
                t = (epoch - self.warmup_epochs) / max(1, self.max_epochs - self.warmup_epochs)
                lr = self.min_lr + 0.5 * (base_lr - self.min_lr) * (1 + math.cos(math.pi * t))
            lrs.append(lr)
        return lrs


# ================== Dataset / Sampler / Collate ==================
class RankDataset(Dataset):
    def __init__(self, code, date, feats, y_clip, y_raw, text_emb: np.ndarray, date_to_gid: Dict[str, int]):
        self.code = code
        self.date = date
        self.feats = feats.astype(np.float32)
        self.y = y_clip.astype(np.float32)
        self.y_raw = y_raw.astype(np.float32)

        self.text_emb = text_emb.astype(np.float32)
        assert self.text_emb.shape[0] == self.feats.shape[0], "text_emb rows != feats rows"
        assert self.text_emb.shape[1] == TEXT_HIDDEN, "text_emb hidden != TEXT_HIDDEN"

        self.date_to_gid = date_to_gid
        self.gid = np.array([self.date_to_gid[d] for d in self.date], dtype=np.int64)

        self.daily_end = DAILY_T * DAILY_F
        self.min_end   = self.daily_end + MIN_T * MIN_F
        assert self.feats.shape[1] == self.min_end, f"feat dim mismatch: {self.feats.shape[1]} vs {self.min_end}"

    def __len__(self):
        return self.feats.shape[0]

    def __getitem__(self, idx):
        f = self.feats[idx]
        daily = f[:self.daily_end].reshape(DAILY_T, DAILY_F)
        minu  = f[self.daily_end:self.min_end].reshape(MIN_T, MIN_F)

        return {
            "daily": torch.from_numpy(daily),
            "minu":  torch.from_numpy(minu),
            "text":  torch.from_numpy(self.text_emb[idx]),
            "y":     torch.tensor(self.y[idx], dtype=torch.float32),
            "y_raw": torch.tensor(self.y_raw[idx], dtype=torch.float32),
            "gid":   torch.tensor(self.gid[idx], dtype=torch.long),
            "code":  self.code[idx],
            "date":  self.date[idx],
        }

class DayBatchSampler(Sampler[List[int]]):
    def __init__(self, groups: Dict[str, np.ndarray], shuffle=True, drop_last=False):
        self.groups = groups
        self.keys = list(groups.keys())
        self.shuffle = shuffle
        self.drop_last = drop_last

    def __iter__(self):
        keys = self.keys[:]
        if self.shuffle:
            random.shuffle(keys)
        for d in keys:
            idxs = self.groups[d]
            if idxs.size == 0:
                continue
            yield idxs.tolist()

    def __len__(self):
        return len(self.keys)

def collate_fn(batch):
    daily = torch.stack([b["daily"] for b in batch], dim=0)
    minu  = torch.stack([b["minu"]  for b in batch], dim=0)
    text  = torch.stack([b["text"]  for b in batch], dim=0)
    y     = torch.stack([b["y"]     for b in batch], dim=0)
    y_raw = torch.stack([b["y_raw"] for b in batch], dim=0)
    gid   = torch.stack([b["gid"]   for b in batch], dim=0)
    codes = [b["code"] for b in batch]
    dates = [b["date"] for b in batch]
    return daily, minu, text, y, y_raw, gid, codes, dates

train_set = RankDataset(tr_code, tr_date, tr_feats, tr_y, tr_y_raw, tr_text, tr_date2gid)
val_set   = RankDataset(va_code, va_date, va_feats, va_y, va_y_raw, va_text, va_date2gid)
test_set  = RankDataset(te_code, te_date, te_feats, te_y, te_y_raw, te_text, te_date2gid)

train_loader = DataLoader(
    train_set,
    batch_sampler=DayBatchSampler(train_groups, shuffle=True),
    num_workers=BATCH_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)
val_loader = DataLoader(
    val_set,
    batch_sampler=DayBatchSampler(val_groups, shuffle=False),
    num_workers=BATCH_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)
test_loader = DataLoader(
    test_set,
    batch_sampler=DayBatchSampler(test_groups, shuffle=False),
    num_workers=BATCH_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)


# ================== 模型（仅保留无门控版本） ==================
class AdditiveAttention(nn.Module):
    def __init__(self, in_dim, hidden_dim=ATTN_HIDDEN):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden_dim)
        self.v    = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, x, mask=None):
        h = torch.tanh(self.proj(x))
        scores = self.v(h).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        w = torch.softmax(scores, dim=-1)
        out = torch.bmm(w.unsqueeze(1), x).squeeze(1)
        return out, w

class DailyEncoder(nn.Module):
    def __init__(self, in_dim=DAILY_F, hidden=LSTM_HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=1, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.attn = AdditiveAttention(2 * hidden, ATTN_HIDDEN)

    def forward(self, x):
        h, _ = self.lstm(x)
        h = self.drop(h)
        out, _ = self.attn(h)
        return out

class MinuteEncoder(nn.Module):
    def __init__(self, in_dim=MIN_F, out_channels=MIN_CONV_C, dropout=DROPOUT):
        super().__init__()
        self.conv1 = nn.Conv1d(in_dim, out_channels, kernel_size=3, padding=1, dilation=1)
        self.bn1   = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=2, dilation=2)
        self.bn2   = nn.BatchNorm1d(out_channels)
        self.conv3 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=4, dilation=4)
        self.bn3   = nn.BatchNorm1d(out_channels)
        self.drop  = nn.Dropout(dropout)
        self.attn  = AdditiveAttention(out_channels, ATTN_HIDDEN)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        h = F.relu(self.bn1(self.conv1(x)))
        h = F.relu(self.bn2(self.conv2(h)))
        h = F.relu(self.bn3(self.conv3(h)))
        h = self.drop(h)
        h = h.permute(0, 2, 1)
        out, _ = self.attn(h)
        return out

class RankModelNoGate(nn.Module):
    def __init__(self, text_hidden: int):
        super().__init__()
        self.daily_enc = DailyEncoder()
        self.min_enc   = MinuteEncoder()

        self.daily_proj = nn.Linear(2 * LSTM_HIDDEN, FUSE_DIM)
        self.min_proj   = nn.Linear(MIN_CONV_C,      FUSE_DIM)
        self.text_proj  = nn.Linear(text_hidden,     FUSE_DIM)

        # 无门控：直接拼接后线性投影
        self.fuse_proj = nn.Linear(3 * FUSE_DIM, FUSE_DIM)
        self.out = nn.Linear(FUSE_DIM, 1)

    def forward(self, daily_x, minu_x, text_x):
        d = self.daily_proj(self.daily_enc(daily_x))
        m = self.min_proj(self.min_enc(minu_x))
        t = self.text_proj(text_x)

        cat = torch.cat([d, m, t], dim=-1)
        fuse = F.relu(self.fuse_proj(cat))

        score = self.out(fuse).squeeze(-1)
        return score


def day_forward_scores(model, daily_x, minu_x, text_x, chunk: Optional[int]):
    B = daily_x.shape[0]
    if not chunk or chunk <= 0 or chunk >= B:
        return model(daily_x, minu_x, text_x)
    outs = []
    for st in range(0, B, chunk):
        ed = min(B, st + chunk)
        outs.append(model(daily_x[st:ed], minu_x[st:ed], text_x[st:ed]))
    return torch.cat(outs, dim=0)


# ================== 排序损失 ==================
def slk_objective(scores: torch.Tensor, labels: torch.Tensor, gids: torch.Tensor,
                  tau_d: float, alpha_entropy: float = 0.0) -> torch.Tensor:
    device = scores.device
    eps = 1e-8
    unique_g = gids.unique()
    loss_sum = torch.tensor(0.0, device=device)
    group_cnt = 0

    for g in unique_g:
        idxs = torch.nonzero(gids == g, as_tuple=False).flatten()
        if idxs.numel() < 2:
            continue

        s = scores[idxs]
        y = labels[idxs]

        tau_y = 1.6
        with torch.no_grad():
            p = torch.softmax(y / max(tau_y, 1e-6), dim=0)
        q = torch.softmax(s / max(tau_d, 1e-6), dim=0)
        ce_loss = -(p * (q + eps).log()).sum()

        loss_g = ce_loss

        neg_mask = (y < 0)
        if neg_mask.any():
            neg_scores = s[neg_mask]
            neg_penalty = torch.relu(neg_scores).mean()
            loss_g = loss_g + 2.0 * neg_penalty

        if alpha_entropy > 0.0:
            ent_q = -(q * (q + eps).log()).sum()
            loss_g = loss_g - alpha_entropy * ent_q

        loss_sum += loss_g
        group_cnt += 1

    if group_cnt == 0:
        return torch.tensor(0.0, device=device)
    return loss_sum / group_cnt


# ================== 评估与导出 ==================
@torch.no_grad()
def infer_scores_full_day(model, loader, use_raw_labels=True):
    model.eval()
    all_scores, all_labels, all_codes, all_dates = [], [], [], []
    for daily_x, minu_x, text_x, y_clip, y_raw, gid, codes, dates in loader:
        daily_x = daily_x.to(DEVICE, non_blocking=True)
        minu_x  = minu_x.to(DEVICE, non_blocking=True)
        text_x  = text_x.to(DEVICE, non_blocking=True)

        s = day_forward_scores(model, daily_x, minu_x, text_x, DAY_FORWARD_CHUNK).detach().cpu().numpy()
        all_scores.append(s)

        lab = (y_raw if use_raw_labels else y_clip).numpy()
        all_labels.append(lab)
        all_codes.extend(codes)
        all_dates.extend(dates)

    return (np.concatenate(all_scores, axis=0),
            np.concatenate(all_labels, axis=0),
            all_codes, all_dates)

def evaluate_daily_precision(scores: np.ndarray, labels: np.ndarray, dates: list,
                             top_list=EVAL_TOP_LIST, thr=0):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    metrics = {k: {"prec": [], "mean_y": []} for k in top_list}
    cover_days = 0

    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        s_day = scores[idxs]
        y_day = labels[idxs]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        cover_days += 1
        for k in top_list:
            k_eff = min(k, s_day.size)
            topk = order[:k_eff]
            yk = y_day[topk]
            hit = (yk >= thr).astype(np.float32)
            metrics[k]["prec"].append(hit.mean() if k_eff > 0 else 0.0)
            metrics[k]["mean_y"].append(yk.mean() if k_eff > 0 else 0.0)

    out = {}
    for k in top_list:
        out[k] = {
            "Precision@K": float(np.mean(metrics[k]["prec"])) if metrics[k]["prec"] else 0.0,
            "AvgReturn@K": float(np.mean(metrics[k]["mean_y"])) if metrics[k]["mean_y"] else 0.0,
            "CoveredDays": int(cover_days),
        }
    return out

def evaluate_daily_ndcg(scores: np.ndarray, labels: np.ndarray, dates: list,
                        top_list=EVAL_TOP_LIST, thr=OPEN_THR):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    def dcg_at_k(rels_sorted_by_rank, k):
        k_eff = min(k, len(rels_sorted_by_rank))
        if k_eff <= 0:
            return 0.0
        gains = (2 ** rels_sorted_by_rank[:k_eff] - 1.0)
        discounts = 1.0 / np.log2(np.arange(2, 2 + k_eff))
        return float(np.sum(gains * discounts))

    bucket = {k: [] for k in top_list}

    for d, idxs in idx_by_day.items():
        arr = np.array(idxs, dtype=np.int64)
        s_day = scores[arr]
        y_day = labels[arr]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        rel = (y_day >= thr).astype(np.int32)
        rel_sorted_gt = np.sort(rel)[::-1]

        for k in top_list:
            k_eff = min(k, s_day.size)
            if k_eff <= 0:
                bucket[k].append(0.0)
                continue
            rel_ranked = rel[order]
            dcg = dcg_at_k(rel_ranked, k_eff)
            idcg = dcg_at_k(rel_sorted_gt, k_eff)
            ndcg = (dcg / idcg) if idcg > 0 else 0.0
            bucket[k].append(ndcg)

    out = {}
    for k in top_list:
        vals = bucket[k]
        out[k] = {
            "NDCG@K": float(np.mean(vals)) if vals else 0.0,
            "CoveredDays": len(vals),
        }
    return out

def export_daily_reco(scores: np.ndarray, labels_raw: np.ndarray, codes: list, dates: list,
                      N=N_TARGET, path_csv="daily_recommendations.csv"):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    rows = []
    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        order = np.argsort(-scores[idxs])
        topn = order[:min(N, idxs.size)]
        for r, j in enumerate(topn, start=1):
            i = idxs[j]
            rows.append({
                "date": dates[i],
                "rank": r,
                "code": codes[i],
                "score": float(scores[i]),
                "true_open_return": float(labels_raw[i])
            })
    pd.DataFrame(rows).sort_values(["date", "rank"]).to_csv(path_csv, index=False)
    print(f"[Export] Saved daily recommendations to {path_csv}")


# ================== Train（仅无门控版本） ==================
seed_all(SEED)
def train():
    model = RankModelNoGate(text_hidden=TEXT_HIDDEN).to(DEVICE)
    best_path = "results/01_multimodal_ranking_model.pth"
    print("[Training Mode] Without Gating (Concat Fusion Only)")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler(DEVICE, enabled=(USE_AMP and DEVICE == "cuda"))
    scheduler = WarmupCosine(optimizer, warmup_epochs=WARMUP_EPOCHS, max_epochs=EPOCHS, min_lr=MIN_LR)

    best_metric = -np.inf

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, days = 0.0, 0

        if epoch <= TAU_ANNEAL_EPS:
            tau_d = TAU_START + (TAU_END - TAU_START) * (epoch - 1) / max(1, TAU_ANNEAL_EPS - 1)
        else:
            tau_d = TAU_END

        for daily_x, minu_x, text_x, y_clip, y_raw, gid, codes, dates in train_loader:
            daily_x = daily_x.to(DEVICE, non_blocking=True)
            minu_x  = minu_x.to(DEVICE, non_blocking=True)
            text_x  = text_x.to(DEVICE, non_blocking=True)
            y_clip  = y_clip.to(DEVICE, non_blocking=True)
            gid     = gid.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if USE_AMP and DEVICE == "cuda":
                with torch.amp.autocast(DEVICE):
                    s = day_forward_scores(model, daily_x, minu_x, text_x, DAY_FORWARD_CHUNK)
                    loss = slk_objective(s, y_clip, gid, tau_d=tau_d, alpha_entropy=ALPHA_ENTROPY)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                s = day_forward_scores(model, daily_x, minu_x, text_x, DAY_FORWARD_CHUNK)
                loss = slk_objective(s, y_clip, gid, tau_d=tau_d, alpha_entropy=ALPHA_ENTROPY)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            total_loss += float(loss.item())
            days += 1

        scheduler.step()
        tr_loss = total_loss / max(1, days)

        scores, labels_raw, codes, dates = infer_scores_full_day(model, val_loader, use_raw_labels=True)

        val_metrics_main = evaluate_daily_precision(
            scores, labels_raw, dates, top_list=[EARLY_STOP_K], thr=OPEN_THR
        )
        val_ndcg_main = evaluate_daily_ndcg(
            scores, labels_raw, dates, top_list=[EARLY_STOP_K], thr=OPEN_THR
        )

        p_main   = val_metrics_main[EARLY_STOP_K]["Precision@K"]
        avg_main = val_metrics_main[EARLY_STOP_K]["AvgReturn@K"]
        ndcg_main = val_ndcg_main[EARLY_STOP_K]["NDCG@K"]

        # ====== 修改点：早停按 AvgReturn@K，而不是 P@1 ======
        val_metric_for_early_stop = p_main

        print(f"[Epoch {epoch}] TrainLoss={tr_loss:.4f} | "
              f"P@{EARLY_STOP_K}(y≥{OPEN_THR})={p_main:.4f}, "
              f"Avg@{EARLY_STOP_K}={avg_main:.4f}, "
              f"NDCG@{EARLY_STOP_K}={ndcg_main:.4f}")

        if val_metric_for_early_stop > best_metric:
            best_metric = val_metric_for_early_stop
            torch.save({
                "model": model.state_dict(),
                "cfg": {
                    "DAILY_T": DAILY_T, "DAILY_F": DAILY_F,
                    "MIN_T": MIN_T, "MIN_F": MIN_F,
                    "TEXT_HIDDEN": TEXT_HIDDEN,
                    "OPEN_THR": OPEN_THR,
                    "TAU_START": TAU_START, "TAU_END": TAU_END,
                    "N_TARGET": N_TARGET,
                    "EARLY_STOP_K": EARLY_STOP_K,
                }
            }, best_path)
            print(f"  >> Saved best to {best_path} "
                  f"(based on P@{EARLY_STOP_K}={val_metric_for_early_stop:.4f})")

    if os.path.exists(best_path):
        ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
        model = RankModelNoGate(text_hidden=TEXT_HIDDEN).to(DEVICE)
        model.load_state_dict(ckpt["model"])
        model.eval()
        print(f"\n[Info] Loaded best checkpoint from {best_path}")

    print("\n[Eval on TEST]")
    scores, labels_raw, codes, dates = infer_scores_full_day(model, test_loader, use_raw_labels=True)
    test_metrics = evaluate_daily_precision(scores, labels_raw, dates, top_list=EVAL_TOP_LIST, thr=OPEN_THR)
    test_ndcg    = evaluate_daily_ndcg(scores, labels_raw, dates, top_list=EVAL_TOP_LIST, thr=OPEN_THR)

    for k in EVAL_TOP_LIST:
        print(f"TEST P@{k}(y≥{OPEN_THR})={test_metrics[k]['Precision@K']:.4f} | "
              f"Avg@{k}={test_metrics[k]['AvgReturn@K']:.4f} | "
              f"NDCG@{k}={test_ndcg[k]['NDCG@K']:.4f} | "
              f"Days={test_metrics[k]['CoveredDays']}")

    export_daily_reco(
        scores, labels_raw, codes, dates,
        N=N_TARGET,
        path_csv="results/01_multimodal_ranking_model.csv"
    )


if __name__ == "__main__":
    train()

/tmp/ipykernel_4072/1225646252.py:95: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_4072/1225646252.py:96: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_4072/1225646252.py:97: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Conside

[Info] Loaded train embeddings: shape=(1610196, 1024) from /root/autodl-tmp/telegram_emb_cache_clean/train_telegram_emb_filtered_filtered.npy
[Info] Loaded val embeddings: shape=(349618, 1024) from /root/autodl-tmp/telegram_emb_cache_clean/val_telegram_emb_filtered_filtered.npy
[Info] Loaded test embeddings: shape=(604975, 1024) from /root/autodl-tmp/telegram_emb_cache_clean/test_telegram_emb_filtered_filtered.npy
[Training Mode] Without Gating (Concat Fusion Only)
[Epoch 1] TrainLoss=7.9741 | P@5(y≥0)=0.5731, Avg@5=0.1943, NDCG@5=0.5739
  >> Saved best to results/01_multimodal_ranking_model.pth (based on P@5=0.5731)
[Epoch 2] TrainLoss=7.9718 | P@5(y≥0)=0.6504, Avg@5=0.8321, NDCG@5=0.6580
  >> Saved best to results/01_multimodal_ranking_model.pth (based on P@5=0.6504)
[Epoch 3] TrainLoss=7.9711 | P@5(y≥0)=0.7395, Avg@5=1.4930, NDCG@5=0.7650
  >> Saved best to results/01_multimodal_ranking_model.pth (based on P@5=0.7395)
[Epoch 4] TrainLoss=7.9705 | P@5(y≥0)=0.5176, Avg@5=0.0420, NDCG@

## Daily + Minute Features

In [2]:
###### exp1_daily_minute_only_no_gate.py
import os
import math
import random
from collections import defaultdict
from typing import List, Dict, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import _LRScheduler
from torch.utils.data import Dataset, DataLoader, Sampler


# ================== 全局与超参 ==================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 数据列规格
DAILY_T, DAILY_F = 5, 43
MIN_T, MIN_F     = 5, 18
FEAT_TOTAL       = DAILY_T * DAILY_F + MIN_T * MIN_F

# 训练批次采样（按“天”为组）
DAY_FORWARD_CHUNK = 4096

BATCH_WORKERS    = 4
PIN_MEMORY       = True

# 训练超参
EPOCHS           = 12
LR               = 1e-4
MIN_LR           = 3e-6
WEIGHT_DECAY     = 1e-4
MAX_GRAD_NORM    = 5.0
# USE_AMP        = True
USE_AMP          = False
WARMUP_EPOCHS    = 2

# 温度退火
TAU_START        = 2.0
TAU_END          = 0.9
TAU_ANNEAL_EPS   = 3

# 模型宽度
LSTM_HIDDEN      = 128
ATTN_HIDDEN      = 64
MIN_CONV_C       = 128
FUSE_DIM         = 128
DROPOUT          = 0.22

# 业务目标
N_TARGET         = 20
OPEN_THR         = 0   # 改为 0

# 熵正则（关闭）
ALPHA_ENTROPY    = 0.0

# 评估
EVAL_TOP_LIST    = [5, 10, 20]

# ========= 新增：早停/选模所依据的 K =========
EARLY_STOP_K     = 5

# ======== 本实验：使用 daily + minute，不使用 text ========
USE_DAILY  = True
USE_MINUTE = True
USE_TEXT   = False

# text 维度仍然保留占位，不改变整体输入接口
TEXT_HIDDEN = 1024


def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # 新版 PyTorch 可加：
    # torch.use_deterministic_algorithms(True)


seed_all()


# ================== 数据加载 ==================
TRAIN_CSV = "/root/autodl-tmp/train_filtered_filtered.csv"
VAL_CSV   = "/root/autodl-tmp/val_filtered_filtered.csv"
TEST_CSV  = "/root/autodl-tmp/test_filtered_filtered.csv"

train_df = pd.read_csv(TRAIN_CSV, low_memory=False)
val_df   = pd.read_csv(VAL_CSV,   low_memory=False)
test_df  = pd.read_csv(TEST_CSV,  low_memory=False)

p1  = train_df["opening_gains"].quantile(0.01)
p99 = train_df["opening_gains"].quantile(0.99)
train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
test_df["opening_gains_clipped"]  = test_df["opening_gains"].clip(lower=p1, upper=p99)


def cast_df(df: pd.DataFrame):
    feat_start = 2
    feat_end   = 2 + FEAT_TOTAL
    feats = df.iloc[:, feat_start:feat_end].astype(np.float32).values
    y_raw  = df["opening_gains"].astype(np.float32).values
    y_clip = df["opening_gains_clipped"].astype(np.float32).values
    code = df.iloc[:, 0].astype(str).values
    date = df.iloc[:, 1].astype(str).values
    return code, date, feats, y_clip, y_raw


tr_code, tr_date, tr_feats, tr_y, tr_y_raw = cast_df(train_df)
va_code, va_date, va_feats, va_y, va_y_raw = cast_df(val_df)
te_code, te_date, te_feats, te_y, te_y_raw = cast_df(test_df)


def build_groups(date_arr: np.ndarray) -> Dict[str, np.ndarray]:
    groups = defaultdict(list)
    for idx, d in enumerate(date_arr):
        groups[d].append(idx)
    return {k: np.array(v, dtype=np.int64) for k, v in groups.items()}


train_groups = build_groups(tr_date)
val_groups   = build_groups(va_date)
test_groups  = build_groups(te_date)

tr_dates = sorted(train_groups.keys())
va_dates = sorted(val_groups.keys())
te_dates = sorted(test_groups.keys())

tr_date2gid = {d: i for i, d in enumerate(tr_dates)}
va_date2gid = {d: i for i, d in enumerate(va_dates)}
te_date2gid = {d: i for i, d in enumerate(te_dates)}


# ================== Scheduler ==================
class WarmupCosine(_LRScheduler):
    def __init__(self, optimizer, warmup_epochs=1, max_epochs=EPOCHS, min_lr=1e-6, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.max_epochs = max_epochs
        self.min_lr = min_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        epoch = self.last_epoch + 1
        lrs = []
        for base_lr in self.base_lrs:
            if epoch <= self.warmup_epochs:
                lr = base_lr * epoch / max(1, self.warmup_epochs)
            else:
                t = (epoch - self.warmup_epochs) / max(1, self.max_epochs - self.warmup_epochs)
                lr = self.min_lr + 0.5 * (base_lr - self.min_lr) * (1 + math.cos(math.pi * t))
            lrs.append(lr)
        return lrs


# ================== Dataset / Sampler / Collate ==================
class RankDataset(Dataset):
    def __init__(self, code, date, feats, y_clip, y_raw, date_to_gid: Dict[str, int]):
        self.code = code
        self.date = date
        self.feats = feats.astype(np.float32)
        self.y = y_clip.astype(np.float32)
        self.y_raw = y_raw.astype(np.float32)

        self.date_to_gid = date_to_gid
        self.gid = np.array([self.date_to_gid[d] for d in self.date], dtype=np.int64)

        self.daily_end = DAILY_T * DAILY_F
        self.min_end   = self.daily_end + MIN_T * MIN_F
        assert self.feats.shape[1] == self.min_end, f"feat dim mismatch: {self.feats.shape[1]} vs {self.min_end}"

        # 全0占位 text
        self.zero_text = np.zeros((TEXT_HIDDEN,), dtype=np.float32)
        self.zero_daily = np.zeros((DAILY_T, DAILY_F), dtype=np.float32)
        self.zero_minu  = np.zeros((MIN_T, MIN_F), dtype=np.float32)

    def __len__(self):
        return self.feats.shape[0]

    def __getitem__(self, idx):
        f = self.feats[idx]
        daily = f[:self.daily_end].reshape(DAILY_T, DAILY_F).astype(np.float32)
        minu  = f[self.daily_end:self.min_end].reshape(MIN_T, MIN_F).astype(np.float32)

        if not USE_DAILY:
            daily = self.zero_daily
        if not USE_MINUTE:
            minu = self.zero_minu

        text = self.zero_text  # 本实验不使用文本（全0）

        return {
            "daily": torch.from_numpy(daily),
            "minu":  torch.from_numpy(minu),
            "text":  torch.from_numpy(text),
            "y":     torch.tensor(self.y[idx], dtype=torch.float32),
            "y_raw": torch.tensor(self.y_raw[idx], dtype=torch.float32),
            "gid":   torch.tensor(self.gid[idx], dtype=torch.long),
            "code":  self.code[idx],
            "date":  self.date[idx],
        }


class DayBatchSampler(Sampler[List[int]]):
    def __init__(self, groups: Dict[str, np.ndarray], shuffle=True, drop_last=False):
        self.groups = groups
        self.keys = list(groups.keys())
        self.shuffle = shuffle
        self.drop_last = drop_last

    def __iter__(self):
        keys = self.keys[:]
        if self.shuffle:
            random.shuffle(keys)
        for d in keys:
            idxs = self.groups[d]
            if idxs.size == 0:
                continue
            yield idxs.tolist()

    def __len__(self):
        return len(self.keys)


def collate_fn(batch):
    daily = torch.stack([b["daily"] for b in batch], dim=0)
    minu  = torch.stack([b["minu"]  for b in batch], dim=0)
    text  = torch.stack([b["text"]  for b in batch], dim=0)
    y     = torch.stack([b["y"]     for b in batch], dim=0)
    y_raw = torch.stack([b["y_raw"] for b in batch], dim=0)
    gid   = torch.stack([b["gid"]   for b in batch], dim=0)
    codes = [b["code"] for b in batch]
    dates = [b["date"] for b in batch]
    return daily, minu, text, y, y_raw, gid, codes, dates


train_set = RankDataset(tr_code, tr_date, tr_feats, tr_y, tr_y_raw, tr_date2gid)
val_set   = RankDataset(va_code, va_date, va_feats, va_y, va_y_raw, va_date2gid)
test_set  = RankDataset(te_code, te_date, te_feats, te_y, te_y_raw, te_date2gid)

train_loader = DataLoader(
    train_set,
    batch_sampler=DayBatchSampler(train_groups, shuffle=True),
    num_workers=BATCH_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)
val_loader = DataLoader(
    val_set,
    batch_sampler=DayBatchSampler(val_groups, shuffle=False),
    num_workers=BATCH_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)
test_loader = DataLoader(
    test_set,
    batch_sampler=DayBatchSampler(test_groups, shuffle=False),
    num_workers=BATCH_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)


# ================== 模型（改为无门控版本） ==================
class AdditiveAttention(nn.Module):
    def __init__(self, in_dim, hidden_dim=ATTN_HIDDEN):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden_dim)
        self.v    = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, x, mask=None):
        h = torch.tanh(self.proj(x))
        scores = self.v(h).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        w = torch.softmax(scores, dim=-1)
        out = torch.bmm(w.unsqueeze(1), x).squeeze(1)
        return out, w


class DailyEncoder(nn.Module):
    def __init__(self, in_dim=DAILY_F, hidden=LSTM_HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=1, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.attn = AdditiveAttention(2 * hidden, ATTN_HIDDEN)

    def forward(self, x):
        h, _ = self.lstm(x)
        h = self.drop(h)
        out, _ = self.attn(h)
        return out  # [B, 2H]


class MinuteEncoder(nn.Module):
    def __init__(self, in_dim=MIN_F, out_channels=MIN_CONV_C, dropout=DROPOUT):
        super().__init__()
        self.conv1 = nn.Conv1d(in_dim, out_channels, kernel_size=3, padding=1, dilation=1)
        self.bn1   = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=2, dilation=2)
        self.bn2   = nn.BatchNorm1d(out_channels)
        self.conv3 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=4, dilation=4)
        self.bn3   = nn.BatchNorm1d(out_channels)
        self.drop  = nn.Dropout(dropout)
        self.attn  = AdditiveAttention(out_channels, ATTN_HIDDEN)

    def forward(self, x):
        x = x.permute(0, 2, 1)  # [B, F, T]
        h = F.relu(self.bn1(self.conv1(x)))
        h = F.relu(self.bn2(self.conv2(h)))
        h = F.relu(self.bn3(self.conv3(h)))
        h = self.drop(h)
        h = h.permute(0, 2, 1)  # [B, T, C]
        out, _ = self.attn(h)
        return out  # [B, C]


class RankModelNoGate(nn.Module):
    def __init__(self, text_hidden: int):
        super().__init__()
        self.daily_enc = DailyEncoder()
        self.min_enc   = MinuteEncoder()

        self.daily_proj = nn.Linear(2 * LSTM_HIDDEN, FUSE_DIM)
        self.min_proj   = nn.Linear(MIN_CONV_C,      FUSE_DIM)
        self.text_proj  = nn.Linear(text_hidden,     FUSE_DIM)

        # 无门控：直接拼接后线性投影
        self.fuse_proj = nn.Linear(3 * FUSE_DIM, FUSE_DIM)
        self.out = nn.Linear(FUSE_DIM, 1)

    def forward(self, daily_x, minu_x, text_x):
        d = self.daily_proj(self.daily_enc(daily_x))
        m = self.min_proj(self.min_enc(minu_x))
        t = self.text_proj(text_x)

        cat = torch.cat([d, m, t], dim=-1)
        fuse = F.relu(self.fuse_proj(cat))

        score = self.out(fuse).squeeze(-1)
        return score


def day_forward_scores(model, daily_x, minu_x, text_x, chunk: Optional[int]):
    B = daily_x.shape[0]
    if not chunk or chunk <= 0 or chunk >= B:
        return model(daily_x, minu_x, text_x)
    outs = []
    for st in range(0, B, chunk):
        ed = min(B, st + chunk)
        outs.append(model(daily_x[st:ed], minu_x[st:ed], text_x[st:ed]))
    return torch.cat(outs, dim=0)


# ================== SL@K loss ==================
def slk_objective(scores: torch.Tensor, labels: torch.Tensor, gids: torch.Tensor,
                  tau_d: float, alpha_entropy: float = 0.0) -> torch.Tensor:
    device = scores.device
    eps = 1e-8
    unique_g = gids.unique()
    loss_sum = torch.tensor(0.0, device=device)
    group_cnt = 0

    for g in unique_g:
        idxs = torch.nonzero(gids == g, as_tuple=False).flatten()
        if idxs.numel() < 2:
            continue

        s = scores[idxs]
        y = labels[idxs]

        tau_y = 1.6
        with torch.no_grad():
            p = torch.softmax(y / max(tau_y, 1e-6), dim=0)
        q = torch.softmax(s / max(tau_d, 1e-6), dim=0)
        ce_loss = -(p * (q + eps).log()).sum()

        loss_g = ce_loss

        neg_mask = (y < 0)
        if neg_mask.any():
            neg_scores = s[neg_mask]
            neg_penalty = torch.relu(neg_scores).mean()
            loss_g = loss_g + 2.0 * neg_penalty

        if alpha_entropy > 0.0:
            ent_q = -(q * (q + eps).log()).sum()
            loss_g = loss_g - alpha_entropy * ent_q

        loss_sum += loss_g
        group_cnt += 1

    if group_cnt == 0:
        return torch.tensor(0.0, device=device)
    return loss_sum / group_cnt


# ================== 评估与导出 ==================
@torch.no_grad()
def infer_scores_full_day(model, loader, use_raw_labels=True):
    model.eval()
    all_scores, all_labels, all_codes, all_dates = [], [], [], []
    for daily_x, minu_x, text_x, y_clip, y_raw, gid, codes, dates in loader:
        daily_x = daily_x.to(DEVICE, non_blocking=True)
        minu_x  = minu_x.to(DEVICE, non_blocking=True)
        text_x  = text_x.to(DEVICE, non_blocking=True)

        s = day_forward_scores(model, daily_x, minu_x, text_x, DAY_FORWARD_CHUNK).detach().cpu().numpy()
        all_scores.append(s)

        lab = (y_raw if use_raw_labels else y_clip).numpy()
        all_labels.append(lab)
        all_codes.extend(codes)
        all_dates.extend(dates)

    return (np.concatenate(all_scores, axis=0),
            np.concatenate(all_labels, axis=0),
            all_codes, all_dates)


def evaluate_daily_precision(scores: np.ndarray, labels: np.ndarray, dates: list,
                             top_list=EVAL_TOP_LIST, thr=0):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    metrics = {k: {"prec": [], "mean_y": []} for k in top_list}
    cover_days = 0

    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        s_day = scores[idxs]
        y_day = labels[idxs]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        cover_days += 1
        for k in top_list:
            k_eff = min(k, s_day.size)
            topk = order[:k_eff]
            yk = y_day[topk]
            hit = (yk >= thr).astype(np.float32)
            metrics[k]["prec"].append(hit.mean() if k_eff > 0 else 0.0)
            metrics[k]["mean_y"].append(yk.mean() if k_eff > 0 else 0.0)

    out = {}
    for k in top_list:
        out[k] = {
            "Precision@K": float(np.mean(metrics[k]["prec"])) if metrics[k]["prec"] else 0.0,
            "AvgReturn@K": float(np.mean(metrics[k]["mean_y"])) if metrics[k]["mean_y"] else 0.0,
            "CoveredDays": int(cover_days),
        }
    return out


def evaluate_daily_ndcg(scores: np.ndarray, labels: np.ndarray, dates: list,
                        top_list=EVAL_TOP_LIST, thr=OPEN_THR):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    def dcg_at_k(rels_sorted_by_rank, k):
        k_eff = min(k, len(rels_sorted_by_rank))
        if k_eff <= 0:
            return 0.0
        gains = (2 ** rels_sorted_by_rank[:k_eff] - 1.0)
        discounts = 1.0 / np.log2(np.arange(2, 2 + k_eff))
        return float(np.sum(gains * discounts))

    bucket = {k: [] for k in top_list}

    for d, idxs in idx_by_day.items():
        arr = np.array(idxs, dtype=np.int64)
        s_day = scores[arr]
        y_day = labels[arr]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        rel = (y_day >= thr).astype(np.int32)
        rel_sorted_gt = np.sort(rel)[::-1]

        for k in top_list:
            k_eff = min(k, s_day.size)
            if k_eff <= 0:
                bucket[k].append(0.0)
                continue
            rel_ranked = rel[order]
            dcg = dcg_at_k(rel_ranked, k_eff)
            idcg = dcg_at_k(rel_sorted_gt, k_eff)
            ndcg = (dcg / idcg) if idcg > 0 else 0.0
            bucket[k].append(ndcg)

    out = {}
    for k in top_list:
        vals = bucket[k]
        out[k] = {
            "NDCG@K": float(np.mean(vals)) if vals else 0.0,
            "CoveredDays": len(vals),
        }
    return out


def export_daily_reco(scores: np.ndarray, labels_raw: np.ndarray, codes: list, dates: list,
                      N=N_TARGET, path_csv="daily_recommendations_exp1_daily_minute_no_gate.csv"):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    rows = []
    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        order = np.argsort(-scores[idxs])
        topn = order[:min(N, idxs.size)]
        for r, j in enumerate(topn, start=1):
            i = idxs[j]
            rows.append({
                "date": dates[i],
                "rank": r,
                "code": codes[i],
                "score": float(scores[i]),
                "true_open_return": float(labels_raw[i])
            })
    pd.DataFrame(rows).sort_values(["date", "rank"]).to_csv(path_csv, index=False)
    print(f"[Export] Saved daily recommendations to {path_csv}")


# ================== Train ==================
seed_all(SEED)
def train():
    model = RankModelNoGate(text_hidden=TEXT_HIDDEN).to(DEVICE)
    print("[Training Mode] Without Gating (Daily + Minute, Text=Zero)")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler(DEVICE, enabled=(USE_AMP and DEVICE == "cuda"))
    scheduler = WarmupCosine(optimizer, warmup_epochs=WARMUP_EPOCHS, max_epochs=EPOCHS, min_lr=MIN_LR)

    best_metric = -np.inf
    best_path = "results/02_daily_minute_features.pth"

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, days = 0.0, 0

        if epoch <= TAU_ANNEAL_EPS:
            tau_d = TAU_START + (TAU_END - TAU_START) * (epoch - 1) / max(1, TAU_ANNEAL_EPS - 1)
        else:
            tau_d = TAU_END

        for daily_x, minu_x, text_x, y_clip, y_raw, gid, codes, dates in train_loader:
            daily_x = daily_x.to(DEVICE, non_blocking=True)
            minu_x  = minu_x.to(DEVICE, non_blocking=True)
            text_x  = text_x.to(DEVICE, non_blocking=True)
            y_clip  = y_clip.to(DEVICE, non_blocking=True)
            gid     = gid.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if USE_AMP and DEVICE == "cuda":
                with torch.amp.autocast(DEVICE):
                    s = day_forward_scores(model, daily_x, minu_x, text_x, DAY_FORWARD_CHUNK)
                    loss = slk_objective(s, y_clip, gid, tau_d=tau_d, alpha_entropy=ALPHA_ENTROPY)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                s = day_forward_scores(model, daily_x, minu_x, text_x, DAY_FORWARD_CHUNK)
                loss = slk_objective(s, y_clip, gid, tau_d=tau_d, alpha_entropy=ALPHA_ENTROPY)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            total_loss += float(loss.item())
            days += 1

        scheduler.step()
        tr_loss = total_loss / max(1, days)

        scores, labels_raw, codes, dates = infer_scores_full_day(model, val_loader, use_raw_labels=True)

        val_metrics_main = evaluate_daily_precision(
            scores, labels_raw, dates, top_list=[EARLY_STOP_K], thr=OPEN_THR
        )
        val_ndcg_main = evaluate_daily_ndcg(
            scores, labels_raw, dates, top_list=[EARLY_STOP_K], thr=OPEN_THR
        )

        p_main    = val_metrics_main[EARLY_STOP_K]["Precision@K"]
        avg_main  = val_metrics_main[EARLY_STOP_K]["AvgReturn@K"]
        ndcg_main = val_ndcg_main[EARLY_STOP_K]["NDCG@K"]

        # ====== 早停按 AvgReturn@K ======
        val_metric_for_early_stop = p_main

        print(f"[Epoch {epoch}] TrainLoss={tr_loss:.4f} | "
              f"P@{EARLY_STOP_K}(y≥{OPEN_THR})={p_main:.4f}, "
              f"Avg@{EARLY_STOP_K}={avg_main:.4f}, "
              f"NDCG@{EARLY_STOP_K}={ndcg_main:.4f}")

        if val_metric_for_early_stop > best_metric:
            best_metric = val_metric_for_early_stop
            torch.save({
                "model": model.state_dict(),
                "cfg": {
                    "DAILY_T": DAILY_T, "DAILY_F": DAILY_F,
                    "MIN_T": MIN_T, "MIN_F": MIN_F,
                    "TEXT_HIDDEN": TEXT_HIDDEN,
                    "OPEN_THR": OPEN_THR,
                    "TAU_START": TAU_START, "TAU_END": TAU_END,
                    "N_TARGET": N_TARGET,
                    "EARLY_STOP_K": EARLY_STOP_K,
                }
            }, best_path)
            print(f"  >> Saved best to {best_path} "
                  f"(based on P@{EARLY_STOP_K}={val_metric_for_early_stop:.4f})")

    if os.path.exists(best_path):
        ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(ckpt["model"])
        model.eval()
        print(f"\n[Info] Loaded best checkpoint from {best_path}")

    print("\n[Eval on TEST]")
    scores, labels_raw, codes, dates = infer_scores_full_day(model, test_loader, use_raw_labels=True)
    test_metrics = evaluate_daily_precision(scores, labels_raw, dates, top_list=EVAL_TOP_LIST, thr=OPEN_THR)
    test_ndcg    = evaluate_daily_ndcg(scores, labels_raw, dates, top_list=EVAL_TOP_LIST, thr=OPEN_THR)

    for k in EVAL_TOP_LIST:
        print(f"TEST P@{k}(y≥{OPEN_THR})={test_metrics[k]['Precision@K']:.4f} | "
              f"Avg@{k}={test_metrics[k]['AvgReturn@K']:.4f} | "
              f"NDCG@{k}={test_ndcg[k]['NDCG@K']:.4f} | "
              f"Days={test_metrics[k]['CoveredDays']}")

    export_daily_reco(
        scores, labels_raw, codes, dates, N=N_TARGET,
        path_csv="results/02_daily_minute_features.csv"
    )


if __name__ == "__main__":
    train()

/tmp/ipykernel_4072/1714503365.py:103: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_4072/1714503365.py:104: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_4072/1714503365.py:105: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

[Training Mode] Without Gating (Daily + Minute, Text=Zero)
[Epoch 1] TrainLoss=7.9742 | P@5(y≥0)=0.5630, Avg@5=0.1259, NDCG@5=0.5636
  >> Saved best to results/02_daily_minute_features.pth (based on P@5=0.5630)
[Epoch 2] TrainLoss=7.9718 | P@5(y≥0)=0.6353, Avg@5=0.7703, NDCG@5=0.6504
  >> Saved best to results/02_daily_minute_features.pth (based on P@5=0.6353)
[Epoch 3] TrainLoss=7.9711 | P@5(y≥0)=0.7496, Avg@5=1.4784, NDCG@5=0.7647
  >> Saved best to results/02_daily_minute_features.pth (based on P@5=0.7496)
[Epoch 4] TrainLoss=7.9705 | P@5(y≥0)=0.5244, Avg@5=-0.1356, NDCG@5=0.5186
[Epoch 5] TrainLoss=7.9698 | P@5(y≥0)=0.5042, Avg@5=-0.3615, NDCG@5=0.4931
[Epoch 6] TrainLoss=7.9693 | P@5(y≥0)=0.4655, Avg@5=-0.5935, NDCG@5=0.4543
[Epoch 7] TrainLoss=7.9690 | P@5(y≥0)=0.4639, Avg@5=-0.6696, NDCG@5=0.4526
[Epoch 8] TrainLoss=7.9688 | P@5(y≥0)=0.4336, Avg@5=-0.9307, NDCG@5=0.4200
[Epoch 9] TrainLoss=7.9685 | P@5(y≥0)=0.4353, Avg@5=-0.8895, NDCG@5=0.4273
[Epoch 10] TrainLoss=7.9684 | P@5(y

## Minute-Only Features

In [3]:
##### exp3_minute_only_no_gate.py
import os
import math
import random
from collections import defaultdict
from typing import List, Dict, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import _LRScheduler
from torch.utils.data import Dataset, DataLoader, Sampler


SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DAILY_T, DAILY_F = 5, 43
MIN_T, MIN_F     = 5, 18
FEAT_TOTAL       = DAILY_T * DAILY_F + MIN_T * MIN_F

TELEGRAM_EMB_DIR = "/root/autodl-tmp/telegram_emb_cache_clean"
DAY_FORWARD_CHUNK = 4096

BATCH_WORKERS    = 4
PIN_MEMORY       = True

EPOCHS           = 12
LR               = 1e-4
MIN_LR           = 3e-6
WEIGHT_DECAY     = 1e-4
MAX_GRAD_NORM    = 5.0
USE_AMP          = False
WARMUP_EPOCHS    = 2

TAU_START        = 2.0
TAU_END          = 0.9
TAU_ANNEAL_EPS   = 3

LSTM_HIDDEN      = 128
ATTN_HIDDEN      = 64
MIN_CONV_C       = 128
FUSE_DIM         = 128
DROPOUT          = 0.22

N_TARGET         = 20
OPEN_THR         = 0
ALPHA_ENTROPY    = 0.0
EVAL_TOP_LIST    = [5, 10, 20]

# ========= 新增：早停/选模所依据的 K =========
EARLY_STOP_K     = 5

# ===== 模态开关（本实验：仅分钟）=====
USE_DAILY = False
USE_MINU  = True
USE_TEXT  = False


def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # 新版 PyTorch 可加：
    # torch.use_deterministic_algorithms(True)

seed_all()


TRAIN_CSV = "/root/autodl-tmp/train_filtered_filtered.csv"
VAL_CSV   = "/root/autodl-tmp/val_filtered_filtered.csv"
TEST_CSV  = "/root/autodl-tmp/test_filtered_filtered.csv"

train_df = pd.read_csv(TRAIN_CSV, low_memory=False)
val_df   = pd.read_csv(VAL_CSV,   low_memory=False)
test_df  = pd.read_csv(TEST_CSV,  low_memory=False)

p1  = train_df["opening_gains"].quantile(0.01)
p99 = train_df["opening_gains"].quantile(0.99)
train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
test_df["opening_gains_clipped"]  = test_df["opening_gains"].clip(lower=p1, upper=p99)

def cast_df(df: pd.DataFrame):
    feat_start = 2
    feat_end   = 2 + FEAT_TOTAL
    feats = df.iloc[:, feat_start:feat_end].astype(np.float32).values
    y_raw  = df["opening_gains"].astype(np.float32).values
    y_clip = df["opening_gains_clipped"].astype(np.float32).values
    code = df.iloc[:, 0].astype(str).values
    date = df.iloc[:, 1].astype(str).values
    return code, date, feats, y_clip, y_raw

tr_code, tr_date, tr_feats, tr_y, tr_y_raw = cast_df(train_df)
va_code, va_date, va_feats, va_y, va_y_raw = cast_df(val_df)
te_code, te_date, te_feats, te_y, te_y_raw = cast_df(test_df)

def build_groups(date_arr: np.ndarray) -> Dict[str, np.ndarray]:
    groups = defaultdict(list)
    for idx, d in enumerate(date_arr):
        groups[d].append(idx)
    return {k: np.array(v, dtype=np.int64) for k, v in groups.items()}

train_groups = build_groups(tr_date)
val_groups   = build_groups(va_date)
test_groups  = build_groups(te_date)

tr_dates = sorted(train_groups.keys())
va_dates = sorted(val_groups.keys())
te_dates = sorted(test_groups.keys())

tr_date2gid = {d: i for i, d in enumerate(tr_dates)}
va_date2gid = {d: i for i, d in enumerate(va_dates)}
te_date2gid = {d: i for i, d in enumerate(te_dates)}

def infer_text_hidden(default_h: int = 1024) -> int:
    for split in ["train", "val", "test"]:
        p = os.path.join(TELEGRAM_EMB_DIR, f"{split}_telegram_emb_filtered_filtered.npy")
        if os.path.exists(p):
            arr = np.load(p, mmap_mode="r")
            if arr.ndim == 2:
                print(f"[Info] Inferred TEXT_HIDDEN={arr.shape[1]} from {p} (mmap)")
                return int(arr.shape[1])
    print(f"[Warn] Cannot find telegram emb files to infer TEXT_HIDDEN. Use default {default_h}")
    return int(default_h)

TEXT_HIDDEN = infer_text_hidden()


class WarmupCosine(_LRScheduler):
    def __init__(self, optimizer, warmup_epochs=1, max_epochs=EPOCHS, min_lr=1e-6, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.max_epochs = max_epochs
        self.min_lr = min_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        epoch = self.last_epoch + 1
        lrs = []
        for base_lr in self.base_lrs:
            if epoch <= self.warmup_epochs:
                lr = base_lr * epoch / max(1, self.warmup_epochs)
            else:
                t = (epoch - self.warmup_epochs) / max(1, self.max_epochs - self.warmup_epochs)
                lr = self.min_lr + 0.5 * (base_lr - self.min_lr) * (1 + math.cos(math.pi * t))
            lrs.append(lr)
        return lrs


class RankDataset(Dataset):
    def __init__(self, code, date, feats, y_clip, y_raw, date_to_gid: Dict[str, int]):
        self.code = code
        self.date = date
        self.feats = feats.astype(np.float32)
        self.y = y_clip.astype(np.float32)
        self.y_raw = y_raw.astype(np.float32)

        self.date_to_gid = date_to_gid
        self.gid = np.array([self.date_to_gid[d] for d in self.date], dtype=np.int64)

        self.daily_end = DAILY_T * DAILY_F
        self.min_end   = self.daily_end + MIN_T * MIN_F
        assert self.feats.shape[1] == self.min_end

    def __len__(self):
        return self.feats.shape[0]

    def __getitem__(self, idx):
        f = self.feats[idx]
        daily = f[:self.daily_end].reshape(DAILY_T, DAILY_F)
        minu  = f[self.daily_end:self.min_end].reshape(MIN_T, MIN_F)
        text = np.zeros((TEXT_HIDDEN,), dtype=np.float32)

        return {
            "daily": torch.from_numpy(daily),
            "minu":  torch.from_numpy(minu),
            "text":  torch.from_numpy(text),
            "y":     torch.tensor(self.y[idx], dtype=torch.float32),
            "y_raw": torch.tensor(self.y_raw[idx], dtype=torch.float32),
            "gid":   torch.tensor(self.gid[idx], dtype=torch.long),
            "code":  self.code[idx],
            "date":  self.date[idx],
        }

class DayBatchSampler(Sampler[List[int]]):
    def __init__(self, groups: Dict[str, np.ndarray], shuffle=True):
        self.groups = groups
        self.keys = list(groups.keys())
        self.shuffle = shuffle

    def __iter__(self):
        keys = self.keys[:]
        if self.shuffle:
            random.shuffle(keys)
        for d in keys:
            idxs = self.groups[d]
            if idxs.size == 0:
                continue
            yield idxs.tolist()

    def __len__(self):
        return len(self.keys)

def collate_fn(batch):
    daily = torch.stack([b["daily"] for b in batch], dim=0)
    minu  = torch.stack([b["minu"]  for b in batch], dim=0)
    text  = torch.stack([b["text"]  for b in batch], dim=0)
    y     = torch.stack([b["y"]     for b in batch], dim=0)
    y_raw = torch.stack([b["y_raw"] for b in batch], dim=0)
    gid   = torch.stack([b["gid"]   for b in batch], dim=0)
    codes = [b["code"] for b in batch]
    dates = [b["date"] for b in batch]

    if not USE_DAILY:
        daily = torch.zeros_like(daily)
    if not USE_MINU:
        minu = torch.zeros_like(minu)
    if not USE_TEXT:
        text = torch.zeros_like(text)

    return daily, minu, text, y, y_raw, gid, codes, dates

train_set = RankDataset(tr_code, tr_date, tr_feats, tr_y, tr_y_raw, tr_date2gid)
val_set   = RankDataset(va_code, va_date, va_feats, va_y, va_y_raw, va_date2gid)
test_set  = RankDataset(te_code, te_date, te_feats, te_y, te_y_raw, te_date2gid)

train_loader = DataLoader(train_set, batch_sampler=DayBatchSampler(train_groups, shuffle=True),
                          num_workers=BATCH_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn)
val_loader   = DataLoader(val_set,   batch_sampler=DayBatchSampler(val_groups, shuffle=False),
                          num_workers=BATCH_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn)
test_loader  = DataLoader(test_set,  batch_sampler=DayBatchSampler(test_groups, shuffle=False),
                          num_workers=BATCH_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn)


class AdditiveAttention(nn.Module):
    def __init__(self, in_dim, hidden_dim=ATTN_HIDDEN):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden_dim)
        self.v    = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, x, mask=None):
        h = torch.tanh(self.proj(x))
        scores = self.v(h).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        w = torch.softmax(scores, dim=-1)
        out = torch.bmm(w.unsqueeze(1), x).squeeze(1)
        return out, w

class DailyEncoder(nn.Module):
    def __init__(self, in_dim=DAILY_F, hidden=LSTM_HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=1, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.attn = AdditiveAttention(2 * hidden, ATTN_HIDDEN)

    def forward(self, x):
        h, _ = self.lstm(x)
        h = self.drop(h)
        out, _ = self.attn(h)
        return out

class MinuteEncoder(nn.Module):
    def __init__(self, in_dim=MIN_F, out_channels=MIN_CONV_C, dropout=DROPOUT):
        super().__init__()
        self.conv1 = nn.Conv1d(in_dim, out_channels, kernel_size=3, padding=1, dilation=1)
        self.bn1   = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=2, dilation=2)
        self.bn2   = nn.BatchNorm1d(out_channels)
        self.conv3 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=4, dilation=4)
        self.bn3   = nn.BatchNorm1d(out_channels)
        self.drop  = nn.Dropout(dropout)
        self.attn  = AdditiveAttention(out_channels, ATTN_HIDDEN)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        h = F.relu(self.bn1(self.conv1(x)))
        h = F.relu(self.bn2(self.conv2(h)))
        h = F.relu(self.bn3(self.conv3(h)))
        h = self.drop(h)
        h = h.permute(0, 2, 1)
        out, _ = self.attn(h)
        return out

class RankModelNoGate(nn.Module):
    def __init__(self, text_hidden: int):
        super().__init__()
        self.daily_enc = DailyEncoder()
        self.min_enc   = MinuteEncoder()

        self.daily_proj = nn.Linear(2 * LSTM_HIDDEN, FUSE_DIM)
        self.min_proj   = nn.Linear(MIN_CONV_C,      FUSE_DIM)
        self.text_proj  = nn.Linear(text_hidden,     FUSE_DIM)

        # 无门控：直接拼接后线性投影
        self.fuse_proj = nn.Linear(3 * FUSE_DIM, FUSE_DIM)
        self.out = nn.Linear(FUSE_DIM, 1)

    def forward(self, daily_x, minu_x, text_x):
        d = self.daily_proj(self.daily_enc(daily_x))
        m = self.min_proj(self.min_enc(minu_x))
        t = self.text_proj(text_x)

        cat = torch.cat([d, m, t], dim=-1)
        fuse = F.relu(self.fuse_proj(cat))

        score = self.out(fuse).squeeze(-1)
        return score

def day_forward_scores(model, daily_x, minu_x, text_x, chunk: Optional[int]):
    B = daily_x.shape[0]
    if not chunk or chunk <= 0 or chunk >= B:
        return model(daily_x, minu_x, text_x)
    outs = []
    for st in range(0, B, chunk):
        ed = min(B, st + chunk)
        outs.append(model(daily_x[st:ed], minu_x[st:ed], text_x[st:ed]))
    return torch.cat(outs, dim=0)


def slk_objective(scores: torch.Tensor, labels: torch.Tensor, gids: torch.Tensor,
                  tau_d: float, alpha_entropy: float = 0.0) -> torch.Tensor:
    device = scores.device
    eps = 1e-8
    unique_g = gids.unique()
    loss_sum = torch.tensor(0.0, device=device)
    group_cnt = 0

    for g in unique_g:
        idxs = torch.nonzero(gids == g, as_tuple=False).flatten()
        if idxs.numel() < 2:
            continue

        s = scores[idxs]
        y = labels[idxs]

        tau_y = 1.6
        with torch.no_grad():
            p = torch.softmax(y / max(tau_y, 1e-6), dim=0)
        q = torch.softmax(s / max(tau_d, 1e-6), dim=0)
        ce_loss = -(p * (q + eps).log()).sum()

        loss_g = ce_loss

        neg_mask = (y < 0)
        if neg_mask.any():
            neg_scores = s[neg_mask]
            neg_penalty = torch.relu(neg_scores).mean()
            loss_g = loss_g + 2.0 * neg_penalty

        if alpha_entropy > 0.0:
            ent_q = -(q * (q + eps).log()).sum()
            loss_g = loss_g - alpha_entropy * ent_q

        loss_sum += loss_g
        group_cnt += 1

    if group_cnt == 0:
        return torch.tensor(0.0, device=device)
    return loss_sum / group_cnt


@torch.no_grad()
def infer_scores_full_day(model, loader, use_raw_labels=True):
    model.eval()
    all_scores, all_labels, all_codes, all_dates = [], [], [], []
    for daily_x, minu_x, text_x, y_clip, y_raw, gid, codes, dates in loader:
        daily_x = daily_x.to(DEVICE, non_blocking=True)
        minu_x  = minu_x.to(DEVICE, non_blocking=True)
        text_x  = text_x.to(DEVICE, non_blocking=True)

        s = day_forward_scores(model, daily_x, minu_x, text_x, DAY_FORWARD_CHUNK).detach().cpu().numpy()
        all_scores.append(s)

        lab = (y_raw if use_raw_labels else y_clip).numpy()
        all_labels.append(lab)
        all_codes.extend(codes)
        all_dates.extend(dates)

    return (np.concatenate(all_scores, axis=0),
            np.concatenate(all_labels, axis=0),
            all_codes, all_dates)

def evaluate_daily_precision(scores: np.ndarray, labels: np.ndarray, dates: list,
                             top_list=EVAL_TOP_LIST, thr=0):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    metrics = {k: {"prec": [], "mean_y": []} for k in top_list}
    cover_days = 0

    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        s_day = scores[idxs]
        y_day = labels[idxs]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        cover_days += 1
        for k in top_list:
            k_eff = min(k, s_day.size)
            topk = order[:k_eff]
            yk = y_day[topk]
            hit = (yk >= thr).astype(np.float32)
            metrics[k]["prec"].append(hit.mean() if k_eff > 0 else 0.0)
            metrics[k]["mean_y"].append(yk.mean() if k_eff > 0 else 0.0)

    out = {}
    for k in top_list:
        out[k] = {
            "Precision@K": float(np.mean(metrics[k]["prec"])) if metrics[k]["prec"] else 0.0,
            "AvgReturn@K": float(np.mean(metrics[k]["mean_y"])) if metrics[k]["mean_y"] else 0.0,
            "CoveredDays": int(cover_days),
        }
    return out

def evaluate_daily_ndcg(scores: np.ndarray, labels: np.ndarray, dates: list,
                        top_list=EVAL_TOP_LIST, thr=OPEN_THR):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    def dcg_at_k(rels_sorted_by_rank, k):
        k_eff = min(k, len(rels_sorted_by_rank))
        if k_eff <= 0:
            return 0.0
        gains = (2 ** rels_sorted_by_rank[:k_eff] - 1.0)
        discounts = 1.0 / np.log2(np.arange(2, 2 + k_eff))
        return float(np.sum(gains * discounts))

    bucket = {k: [] for k in top_list}

    for d, idxs in idx_by_day.items():
        arr = np.array(idxs, dtype=np.int64)
        s_day = scores[arr]
        y_day = labels[arr]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        rel = (y_day >= thr).astype(np.int32)
        rel_sorted_gt = np.sort(rel)[::-1]

        for k in top_list:
            k_eff = min(k, s_day.size)
            if k_eff <= 0:
                bucket[k].append(0.0)
                continue
            rel_ranked = rel[order]
            dcg = dcg_at_k(rel_ranked, k_eff)
            idcg = dcg_at_k(rel_sorted_gt, k_eff)
            ndcg = (dcg / idcg) if idcg > 0 else 0.0
            bucket[k].append(ndcg)

    out = {}
    for k in top_list:
        vals = bucket[k]
        out[k] = {"NDCG@K": float(np.mean(vals)) if vals else 0.0, "CoveredDays": len(vals)}
    return out

def export_daily_reco(scores: np.ndarray, labels_raw: np.ndarray, codes: list, dates: list,
                      N=N_TARGET, path_csv="daily_recommendations.csv"):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    rows = []
    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        order = np.argsort(-scores[idxs])
        topn = order[:min(N, idxs.size)]
        for r, j in enumerate(topn, start=1):
            i = idxs[j]
            rows.append({
                "date": dates[i],
                "rank": r,
                "code": codes[i],
                "score": float(scores[i]),
                "true_open_return": float(labels_raw[i])
            })
    pd.DataFrame(rows).sort_values(["date", "rank"]).to_csv(path_csv, index=False)
    print(f"[Export] Saved daily recommendations to {path_csv}")

def autocast_context():
    if USE_AMP and DEVICE == "cuda":
        try:
            return torch.amp.autocast(device_type="cuda")
        except Exception:
            return torch.cuda.amp.autocast()
    class Dummy:
        def __enter__(self): return None
        def __exit__(self, exc_type, exc, tb): return False
    return Dummy()

def make_scaler():
    if USE_AMP and DEVICE == "cuda":
        try:
            return torch.amp.GradScaler("cuda")
        except Exception:
            return torch.cuda.amp.GradScaler()
    class DummyScaler:
        def scale(self, x): return x
        def unscale_(self, _): pass
        def step(self, opt): opt.step()
        def update(self): pass
    return DummyScaler()

seed_all(SEED)
def train():
    model = RankModelNoGate(text_hidden=TEXT_HIDDEN).to(DEVICE)
    print("[Run] Experiment 3: Minute only (Daily/Text zero, no gating)")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = make_scaler()
    scheduler = WarmupCosine(optimizer, warmup_epochs=WARMUP_EPOCHS, max_epochs=EPOCHS, min_lr=MIN_LR)

    best_metric = -np.inf
    best_path = "results/03_minute_only_features.pth"

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, days = 0.0, 0

        tau_d = (TAU_START + (TAU_END - TAU_START) * (epoch - 1) / max(1, TAU_ANNEAL_EPS - 1)) if epoch <= TAU_ANNEAL_EPS else TAU_END

        for daily_x, minu_x, text_x, y_clip, y_raw, gid, codes, dates in train_loader:
            daily_x = daily_x.to(DEVICE, non_blocking=True)
            minu_x  = minu_x.to(DEVICE, non_blocking=True)
            text_x  = text_x.to(DEVICE, non_blocking=True)
            y_clip  = y_clip.to(DEVICE, non_blocking=True)
            gid     = gid.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                s = day_forward_scores(model, daily_x, minu_x, text_x, DAY_FORWARD_CHUNK)
                loss = slk_objective(s, y_clip, gid, tau_d=tau_d, alpha_entropy=ALPHA_ENTROPY)

            if USE_AMP and DEVICE == "cuda":
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            total_loss += float(loss.item())
            days += 1

        scheduler.step()
        tr_loss = total_loss / max(1, days)

        scores, labels_raw, codes, dates = infer_scores_full_day(model, val_loader, use_raw_labels=True)

        val_metrics_main = evaluate_daily_precision(
            scores, labels_raw, dates, top_list=[EARLY_STOP_K], thr=OPEN_THR
        )
        val_ndcg_main = evaluate_daily_ndcg(
            scores, labels_raw, dates, top_list=[EARLY_STOP_K], thr=OPEN_THR
        )

        p_main   = val_metrics_main[EARLY_STOP_K]["Precision@K"]
        avg_main = val_metrics_main[EARLY_STOP_K]["AvgReturn@K"]
        ndcg_main = val_ndcg_main[EARLY_STOP_K]["NDCG@K"]

        # ====== 早停按 AvgReturn@K，而不是 P@1 ======
        val_metric_for_early_stop = p_main

        print(f"[Epoch {epoch}] TrainLoss={tr_loss:.4f} | "
              f"P@{EARLY_STOP_K}(y≥{OPEN_THR})={p_main:.4f}, "
              f"Avg@{EARLY_STOP_K}={avg_main:.4f}, "
              f"NDCG@{EARLY_STOP_K}={ndcg_main:.4f}")

        if val_metric_for_early_stop > best_metric:
            best_metric = val_metric_for_early_stop
            torch.save({
                "model": model.state_dict(),
                "cfg": {
                    "DAILY_T": DAILY_T, "DAILY_F": DAILY_F,
                    "MIN_T": MIN_T, "MIN_F": MIN_F,
                    "TEXT_HIDDEN": TEXT_HIDDEN,
                    "OPEN_THR": OPEN_THR,
                    "TAU_START": TAU_START, "TAU_END": TAU_END,
                    "N_TARGET": N_TARGET,
                    "EARLY_STOP_K": EARLY_STOP_K,
                }
            }, best_path)
            print(f"  >> Saved best to {best_path} "
                  f"(based on P@{EARLY_STOP_K}={val_metric_for_early_stop:.4f})")

    if os.path.exists(best_path):
        ckpt = torch.load(best_path, map_location=DEVICE)
        model = RankModelNoGate(text_hidden=TEXT_HIDDEN).to(DEVICE)
        model.load_state_dict(ckpt["model"])
        model.eval()
        print(f"\n[Info] Loaded best checkpoint from {best_path}")

    print("\n[Eval on TEST]")
    scores, labels_raw, codes, dates = infer_scores_full_day(model, test_loader, use_raw_labels=True)
    test_metrics = evaluate_daily_precision(scores, labels_raw, dates, top_list=EVAL_TOP_LIST, thr=OPEN_THR)
    test_ndcg    = evaluate_daily_ndcg(scores, labels_raw, dates, top_list=EVAL_TOP_LIST, thr=OPEN_THR)

    for k in EVAL_TOP_LIST:
        print(f"TEST P@{k}(y≥{OPEN_THR})={test_metrics[k]['Precision@K']:.4f} | "
              f"Avg@{k}={test_metrics[k]['AvgReturn@K']:.4f} | "
              f"NDCG@{k}={test_ndcg[k]['NDCG@K']:.4f}")

    export_daily_reco(
        scores, labels_raw, codes, dates,
        N=N_TARGET,
        path_csv="results/03_minute_only_features.csv"
    )


if __name__ == "__main__":
    train()

/tmp/ipykernel_4072/783817753.py:87: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_4072/783817753.py:88: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_4072/783817753.py:89: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider j

[Info] Inferred TEXT_HIDDEN=1024 from /root/autodl-tmp/telegram_emb_cache_clean/train_telegram_emb_filtered_filtered.npy (mmap)
[Run] Experiment 3: Minute only (Daily/Text zero, no gating)
[Epoch 1] TrainLoss=7.9741 | P@5(y≥0)=0.6017, Avg@5=0.4445, NDCG@5=0.5981
  >> Saved best to results/03_minute_only_features.pth (based on P@5=0.6017)
[Epoch 2] TrainLoss=7.9720 | P@5(y≥0)=0.6807, Avg@5=1.2573, NDCG@5=0.7142
  >> Saved best to results/03_minute_only_features.pth (based on P@5=0.6807)
[Epoch 3] TrainLoss=7.9715 | P@5(y≥0)=0.7210, Avg@5=1.4652, NDCG@5=0.7516
  >> Saved best to results/03_minute_only_features.pth (based on P@5=0.7210)
[Epoch 4] TrainLoss=7.9710 | P@5(y≥0)=0.5916, Avg@5=-0.0057, NDCG@5=0.5977
[Epoch 5] TrainLoss=7.9703 | P@5(y≥0)=0.5664, Avg@5=-0.0617, NDCG@5=0.5743
[Epoch 6] TrainLoss=7.9699 | P@5(y≥0)=0.5882, Avg@5=-0.0799, NDCG@5=0.5953
[Epoch 7] TrainLoss=7.9697 | P@5(y≥0)=0.5765, Avg@5=-0.1115, NDCG@5=0.5907
[Epoch 8] TrainLoss=7.9695 | P@5(y≥0)=0.5950, Avg@5=-0.163

## Daily-Only Features

In [4]:
##### exp4_daily_only_no_gate.py
import os
import math
import random
from collections import defaultdict
from typing import List, Dict, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import _LRScheduler
from torch.utils.data import Dataset, DataLoader, Sampler


SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DAILY_T, DAILY_F = 5, 43
MIN_T, MIN_F     = 5, 18
FEAT_TOTAL       = DAILY_T * DAILY_F + MIN_T * MIN_F

TELEGRAM_EMB_DIR = "/root/autodl-tmp/telegram_emb_cache_clean"
DAY_FORWARD_CHUNK = 4096

BATCH_WORKERS    = 4
PIN_MEMORY       = True

EPOCHS           = 12
LR               = 1e-4
MIN_LR           = 3e-6
WEIGHT_DECAY     = 1e-4
MAX_GRAD_NORM    = 5.0
USE_AMP          = False
WARMUP_EPOCHS    = 2

TAU_START        = 2.0
TAU_END          = 0.9
TAU_ANNEAL_EPS   = 3

LSTM_HIDDEN      = 128
ATTN_HIDDEN      = 64
MIN_CONV_C       = 128
FUSE_DIM         = 128
DROPOUT          = 0.22

N_TARGET         = 20
OPEN_THR         = 0
ALPHA_ENTROPY    = 0.0
EVAL_TOP_LIST    = [5, 10, 20]

# ========= 新增：早停/选模所依据的 K =========
EARLY_STOP_K     = 5

# ===== 模态开关（本实验：仅日频）=====
USE_DAILY = True
USE_MINU  = False
USE_TEXT  = False


def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # 新版 PyTorch 可加：
    # torch.use_deterministic_algorithms(True)

seed_all()


TRAIN_CSV = "/root/autodl-tmp/train_filtered_filtered.csv"
VAL_CSV   = "/root/autodl-tmp/val_filtered_filtered.csv"
TEST_CSV  = "/root/autodl-tmp/test_filtered_filtered.csv"

train_df = pd.read_csv(TRAIN_CSV, low_memory=False)
val_df   = pd.read_csv(VAL_CSV,   low_memory=False)
test_df  = pd.read_csv(TEST_CSV,  low_memory=False)

p1  = train_df["opening_gains"].quantile(0.01)
p99 = train_df["opening_gains"].quantile(0.99)
train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
test_df["opening_gains_clipped"]  = test_df["opening_gains"].clip(lower=p1, upper=p99)

def cast_df(df: pd.DataFrame):
    feat_start = 2
    feat_end   = 2 + FEAT_TOTAL
    feats = df.iloc[:, feat_start:feat_end].astype(np.float32).values
    y_raw  = df["opening_gains"].astype(np.float32).values
    y_clip = df["opening_gains_clipped"].astype(np.float32).values
    code = df.iloc[:, 0].astype(str).values
    date = df.iloc[:, 1].astype(str).values
    return code, date, feats, y_clip, y_raw

tr_code, tr_date, tr_feats, tr_y, tr_y_raw = cast_df(train_df)
va_code, va_date, va_feats, va_y, va_y_raw = cast_df(val_df)
te_code, te_date, te_feats, te_y, te_y_raw = cast_df(test_df)

def build_groups(date_arr: np.ndarray) -> Dict[str, np.ndarray]:
    groups = defaultdict(list)
    for idx, d in enumerate(date_arr):
        groups[d].append(idx)
    return {k: np.array(v, dtype=np.int64) for k, v in groups.items()}

train_groups = build_groups(tr_date)
val_groups   = build_groups(va_date)
test_groups  = build_groups(te_date)

tr_dates = sorted(train_groups.keys())
va_dates = sorted(val_groups.keys())
te_dates = sorted(test_groups.keys())

tr_date2gid = {d: i for i, d in enumerate(tr_dates)}
va_date2gid = {d: i for i, d in enumerate(va_dates)}
te_date2gid = {d: i for i, d in enumerate(te_dates)}

def infer_text_hidden(default_h: int = 1024) -> int:
    for split in ["train", "val", "test"]:
        p = os.path.join(TELEGRAM_EMB_DIR, f"{split}_telegram_emb_filtered_filtered.npy")
        if os.path.exists(p):
            arr = np.load(p, mmap_mode="r")
            if arr.ndim == 2:
                print(f"[Info] Inferred TEXT_HIDDEN={arr.shape[1]} from {p} (mmap)")
                return int(arr.shape[1])
    print(f"[Warn] Cannot find telegram emb files to infer TEXT_HIDDEN. Use default {default_h}")
    return int(default_h)

TEXT_HIDDEN = infer_text_hidden()


class WarmupCosine(_LRScheduler):
    def __init__(self, optimizer, warmup_epochs=1, max_epochs=EPOCHS, min_lr=1e-6, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.max_epochs = max_epochs
        self.min_lr = min_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        epoch = self.last_epoch + 1
        lrs = []
        for base_lr in self.base_lrs:
            if epoch <= self.warmup_epochs:
                lr = base_lr * epoch / max(1, self.warmup_epochs)
            else:
                t = (epoch - self.warmup_epochs) / max(1, self.max_epochs - self.warmup_epochs)
                lr = self.min_lr + 0.5 * (base_lr - self.min_lr) * (1 + math.cos(math.pi * t))
            lrs.append(lr)
        return lrs


class RankDataset(Dataset):
    def __init__(self, code, date, feats, y_clip, y_raw, date_to_gid: Dict[str, int]):
        self.code = code
        self.date = date
        self.feats = feats.astype(np.float32)
        self.y = y_clip.astype(np.float32)
        self.y_raw = y_raw.astype(np.float32)

        self.date_to_gid = date_to_gid
        self.gid = np.array([self.date_to_gid[d] for d in self.date], dtype=np.int64)

        self.daily_end = DAILY_T * DAILY_F
        self.min_end   = self.daily_end + MIN_T * MIN_F
        assert self.feats.shape[1] == self.min_end

    def __len__(self):
        return self.feats.shape[0]

    def __getitem__(self, idx):
        f = self.feats[idx]
        daily = f[:self.daily_end].reshape(DAILY_T, DAILY_F)
        minu  = f[self.daily_end:self.min_end].reshape(MIN_T, MIN_F)
        text = np.zeros((TEXT_HIDDEN,), dtype=np.float32)

        return {
            "daily": torch.from_numpy(daily),
            "minu":  torch.from_numpy(minu),
            "text":  torch.from_numpy(text),
            "y":     torch.tensor(self.y[idx], dtype=torch.float32),
            "y_raw": torch.tensor(self.y_raw[idx], dtype=torch.float32),
            "gid":   torch.tensor(self.gid[idx], dtype=torch.long),
            "code":  self.code[idx],
            "date":  self.date[idx],
        }

class DayBatchSampler(Sampler[List[int]]):
    def __init__(self, groups: Dict[str, np.ndarray], shuffle=True):
        self.groups = groups
        self.keys = list(groups.keys())
        self.shuffle = shuffle

    def __iter__(self):
        keys = self.keys[:]
        if self.shuffle:
            random.shuffle(keys)
        for d in keys:
            idxs = self.groups[d]
            if idxs.size == 0:
                continue
            yield idxs.tolist()

    def __len__(self):
        return len(self.keys)

def collate_fn(batch):
    daily = torch.stack([b["daily"] for b in batch], dim=0)
    minu  = torch.stack([b["minu"]  for b in batch], dim=0)
    text  = torch.stack([b["text"]  for b in batch], dim=0)
    y     = torch.stack([b["y"]     for b in batch], dim=0)
    y_raw = torch.stack([b["y_raw"] for b in batch], dim=0)
    gid   = torch.stack([b["gid"]   for b in batch], dim=0)
    codes = [b["code"] for b in batch]
    dates = [b["date"] for b in batch]

    if not USE_DAILY:
        daily = torch.zeros_like(daily)
    if not USE_MINU:
        minu = torch.zeros_like(minu)
    if not USE_TEXT:
        text = torch.zeros_like(text)

    return daily, minu, text, y, y_raw, gid, codes, dates

train_set = RankDataset(tr_code, tr_date, tr_feats, tr_y, tr_y_raw, tr_date2gid)
val_set   = RankDataset(va_code, va_date, va_feats, va_y, va_y_raw, va_date2gid)
test_set  = RankDataset(te_code, te_date, te_feats, te_y, te_y_raw, te_date2gid)

train_loader = DataLoader(train_set, batch_sampler=DayBatchSampler(train_groups, shuffle=True),
                          num_workers=BATCH_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn)
val_loader   = DataLoader(val_set,   batch_sampler=DayBatchSampler(val_groups, shuffle=False),
                          num_workers=BATCH_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn)
test_loader  = DataLoader(test_set,  batch_sampler=DayBatchSampler(test_groups, shuffle=False),
                          num_workers=BATCH_WORKERS, pin_memory=PIN_MEMORY, collate_fn=collate_fn)


class AdditiveAttention(nn.Module):
    def __init__(self, in_dim, hidden_dim=ATTN_HIDDEN):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden_dim)
        self.v    = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, x, mask=None):
        h = torch.tanh(self.proj(x))
        scores = self.v(h).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        w = torch.softmax(scores, dim=-1)
        out = torch.bmm(w.unsqueeze(1), x).squeeze(1)
        return out, w

class DailyEncoder(nn.Module):
    def __init__(self, in_dim=DAILY_F, hidden=LSTM_HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=1, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.attn = AdditiveAttention(2 * hidden, ATTN_HIDDEN)

    def forward(self, x):
        h, _ = self.lstm(x)
        h = self.drop(h)
        out, _ = self.attn(h)
        return out

class MinuteEncoder(nn.Module):
    def __init__(self, in_dim=MIN_F, out_channels=MIN_CONV_C, dropout=DROPOUT):
        super().__init__()
        self.conv1 = nn.Conv1d(in_dim, out_channels, kernel_size=3, padding=1, dilation=1)
        self.bn1   = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=2, dilation=2)
        self.bn2   = nn.BatchNorm1d(out_channels)
        self.conv3 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=4, dilation=4)
        self.bn3   = nn.BatchNorm1d(out_channels)
        self.drop  = nn.Dropout(dropout)
        self.attn  = AdditiveAttention(out_channels, ATTN_HIDDEN)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        h = F.relu(self.bn1(self.conv1(x)))
        h = F.relu(self.bn2(self.conv2(h)))
        h = F.relu(self.bn3(self.conv3(h)))
        h = self.drop(h)
        h = h.permute(0, 2, 1)
        out, _ = self.attn(h)
        return out

class RankModelNoGate(nn.Module):
    def __init__(self, text_hidden: int):
        super().__init__()
        self.daily_enc = DailyEncoder()
        self.min_enc   = MinuteEncoder()

        self.daily_proj = nn.Linear(2 * LSTM_HIDDEN, FUSE_DIM)
        self.min_proj   = nn.Linear(MIN_CONV_C,      FUSE_DIM)
        self.text_proj  = nn.Linear(text_hidden,     FUSE_DIM)

        # 无门控：直接拼接后线性投影
        self.fuse_proj = nn.Linear(3 * FUSE_DIM, FUSE_DIM)
        self.out = nn.Linear(FUSE_DIM, 1)

    def forward(self, daily_x, minu_x, text_x):
        d = self.daily_proj(self.daily_enc(daily_x))
        m = self.min_proj(self.min_enc(minu_x))
        t = self.text_proj(text_x)

        cat = torch.cat([d, m, t], dim=-1)
        fuse = F.relu(self.fuse_proj(cat))

        score = self.out(fuse).squeeze(-1)
        return score

def day_forward_scores(model, daily_x, minu_x, text_x, chunk: Optional[int]):
    B = daily_x.shape[0]
    if not chunk or chunk <= 0 or chunk >= B:
        return model(daily_x, minu_x, text_x)
    outs = []
    for st in range(0, B, chunk):
        ed = min(B, st + chunk)
        outs.append(model(daily_x[st:ed], minu_x[st:ed], text_x[st:ed]))
    return torch.cat(outs, dim=0)


def slk_objective(scores: torch.Tensor, labels: torch.Tensor, gids: torch.Tensor,
                  tau_d: float, alpha_entropy: float = 0.0) -> torch.Tensor:
    device = scores.device
    eps = 1e-8
    unique_g = gids.unique()
    loss_sum = torch.tensor(0.0, device=device)
    group_cnt = 0

    for g in unique_g:
        idxs = torch.nonzero(gids == g, as_tuple=False).flatten()
        if idxs.numel() < 2:
            continue

        s = scores[idxs]
        y = labels[idxs]

        tau_y = 1.6
        with torch.no_grad():
            p = torch.softmax(y / max(tau_y, 1e-6), dim=0)
        q = torch.softmax(s / max(tau_d, 1e-6), dim=0)
        ce_loss = -(p * (q + eps).log()).sum()

        loss_g = ce_loss

        neg_mask = (y < 0)
        if neg_mask.any():
            neg_scores = s[neg_mask]
            neg_penalty = torch.relu(neg_scores).mean()
            loss_g = loss_g + 2.0 * neg_penalty

        if alpha_entropy > 0.0:
            ent_q = -(q * (q + eps).log()).sum()
            loss_g = loss_g - alpha_entropy * ent_q

        loss_sum += loss_g
        group_cnt += 1

    if group_cnt == 0:
        return torch.tensor(0.0, device=device)
    return loss_sum / group_cnt


@torch.no_grad()
def infer_scores_full_day(model, loader, use_raw_labels=True):
    model.eval()
    all_scores, all_labels, all_codes, all_dates = [], [], [], []
    for daily_x, minu_x, text_x, y_clip, y_raw, gid, codes, dates in loader:
        daily_x = daily_x.to(DEVICE, non_blocking=True)
        minu_x  = minu_x.to(DEVICE, non_blocking=True)
        text_x  = text_x.to(DEVICE, non_blocking=True)

        s = day_forward_scores(model, daily_x, minu_x, text_x, DAY_FORWARD_CHUNK).detach().cpu().numpy()
        all_scores.append(s)

        lab = (y_raw if use_raw_labels else y_clip).numpy()
        all_labels.append(lab)
        all_codes.extend(codes)
        all_dates.extend(dates)

    return (np.concatenate(all_scores, axis=0),
            np.concatenate(all_labels, axis=0),
            all_codes, all_dates)

def evaluate_daily_precision(scores: np.ndarray, labels: np.ndarray, dates: list,
                             top_list=EVAL_TOP_LIST, thr=0):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    metrics = {k: {"prec": [], "mean_y": []} for k in top_list}
    cover_days = 0

    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        s_day = scores[idxs]
        y_day = labels[idxs]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        cover_days += 1
        for k in top_list:
            k_eff = min(k, s_day.size)
            topk = order[:k_eff]
            yk = y_day[topk]
            hit = (yk >= thr).astype(np.float32)
            metrics[k]["prec"].append(hit.mean() if k_eff > 0 else 0.0)
            metrics[k]["mean_y"].append(yk.mean() if k_eff > 0 else 0.0)

    out = {}
    for k in top_list:
        out[k] = {
            "Precision@K": float(np.mean(metrics[k]["prec"])) if metrics[k]["prec"] else 0.0,
            "AvgReturn@K": float(np.mean(metrics[k]["mean_y"])) if metrics[k]["mean_y"] else 0.0,
            "CoveredDays": int(cover_days),
        }
    return out

def evaluate_daily_ndcg(scores: np.ndarray, labels: np.ndarray, dates: list,
                        top_list=EVAL_TOP_LIST, thr=OPEN_THR):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    def dcg_at_k(rels_sorted_by_rank, k):
        k_eff = min(k, len(rels_sorted_by_rank))
        if k_eff <= 0:
            return 0.0
        gains = (2 ** rels_sorted_by_rank[:k_eff] - 1.0)
        discounts = 1.0 / np.log2(np.arange(2, 2 + k_eff))
        return float(np.sum(gains * discounts))

    bucket = {k: [] for k in top_list}

    for d, idxs in idx_by_day.items():
        arr = np.array(idxs, dtype=np.int64)
        s_day = scores[arr]
        y_day = labels[arr]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        rel = (y_day >= thr).astype(np.int32)
        rel_sorted_gt = np.sort(rel)[::-1]

        for k in top_list:
            k_eff = min(k, s_day.size)
            if k_eff <= 0:
                bucket[k].append(0.0)
                continue
            rel_ranked = rel[order]
            dcg = dcg_at_k(rel_ranked, k_eff)
            idcg = dcg_at_k(rel_sorted_gt, k_eff)
            ndcg = (dcg / idcg) if idcg > 0 else 0.0
            bucket[k].append(ndcg)

    out = {}
    for k in top_list:
        vals = bucket[k]
        out[k] = {"NDCG@K": float(np.mean(vals)) if vals else 0.0, "CoveredDays": len(vals)}
    return out

def export_daily_reco(scores: np.ndarray, labels_raw: np.ndarray, codes: list, dates: list,
                      N=N_TARGET, path_csv="daily_recommendations.csv"):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    rows = []
    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        order = np.argsort(-scores[idxs])
        topn = order[:min(N, idxs.size)]
        for r, j in enumerate(topn, start=1):
            i = idxs[j]
            rows.append({
                "date": dates[i],
                "rank": r,
                "code": codes[i],
                "score": float(scores[i]),
                "true_open_return": float(labels_raw[i])
            })
    pd.DataFrame(rows).sort_values(["date", "rank"]).to_csv(path_csv, index=False)
    print(f"[Export] Saved daily recommendations to {path_csv}")

def autocast_context():
    if USE_AMP and DEVICE == "cuda":
        try:
            return torch.amp.autocast(device_type="cuda")
        except Exception:
            return torch.cuda.amp.autocast()
    class Dummy:
        def __enter__(self): return None
        def __exit__(self, exc_type, exc, tb): return False
    return Dummy()

def make_scaler():
    if USE_AMP and DEVICE == "cuda":
        try:
            return torch.amp.GradScaler("cuda")
        except Exception:
            return torch.cuda.amp.GradScaler()
    class DummyScaler:
        def scale(self, x): return x
        def unscale_(self, _): pass
        def step(self, opt): opt.step()
        def update(self): pass
    return DummyScaler()

seed_all(SEED)
def train():
    model = RankModelNoGate(text_hidden=TEXT_HIDDEN).to(DEVICE)
    print("[Run] Experiment 4: Daily only (Minute/Text zero, no gating)")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = make_scaler()
    scheduler = WarmupCosine(optimizer, warmup_epochs=WARMUP_EPOCHS, max_epochs=EPOCHS, min_lr=MIN_LR)

    best_metric = -np.inf
    best_path = "results/04_daily_only_features.pth"

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, days = 0.0, 0

        tau_d = (TAU_START + (TAU_END - TAU_START) * (epoch - 1) / max(1, TAU_ANNEAL_EPS - 1)) if epoch <= TAU_ANNEAL_EPS else TAU_END

        for daily_x, minu_x, text_x, y_clip, y_raw, gid, codes, dates in train_loader:
            daily_x = daily_x.to(DEVICE, non_blocking=True)
            minu_x  = minu_x.to(DEVICE, non_blocking=True)
            text_x  = text_x.to(DEVICE, non_blocking=True)
            y_clip  = y_clip.to(DEVICE, non_blocking=True)
            gid     = gid.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                s = day_forward_scores(model, daily_x, minu_x, text_x, DAY_FORWARD_CHUNK)
                loss = slk_objective(s, y_clip, gid, tau_d=tau_d, alpha_entropy=ALPHA_ENTROPY)

            if USE_AMP and DEVICE == "cuda":
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            total_loss += float(loss.item())
            days += 1

        scheduler.step()
        tr_loss = total_loss / max(1, days)

        scores, labels_raw, codes, dates = infer_scores_full_day(model, val_loader, use_raw_labels=True)

        val_metrics_main = evaluate_daily_precision(
            scores, labels_raw, dates, top_list=[EARLY_STOP_K], thr=OPEN_THR
        )
        val_ndcg_main = evaluate_daily_ndcg(
            scores, labels_raw, dates, top_list=[EARLY_STOP_K], thr=OPEN_THR
        )

        p_main   = val_metrics_main[EARLY_STOP_K]["Precision@K"]
        avg_main = val_metrics_main[EARLY_STOP_K]["AvgReturn@K"]
        ndcg_main = val_ndcg_main[EARLY_STOP_K]["NDCG@K"]

        # ====== 修改点：早停按 AvgReturn@K，而不是 P@1 ======
        val_metric_for_early_stop = p_main

        print(f"[Epoch {epoch}] TrainLoss={tr_loss:.4f} | "
              f"P@{EARLY_STOP_K}(y≥{OPEN_THR})={p_main:.4f}, "
              f"Avg@{EARLY_STOP_K}={avg_main:.4f}, "
              f"NDCG@{EARLY_STOP_K}={ndcg_main:.4f}")

        if val_metric_for_early_stop > best_metric:
            best_metric = val_metric_for_early_stop
            torch.save({
                "model": model.state_dict(),
                "cfg": {
                    "DAILY_T": DAILY_T, "DAILY_F": DAILY_F,
                    "MIN_T": MIN_T, "MIN_F": MIN_F,
                    "TEXT_HIDDEN": TEXT_HIDDEN,
                    "OPEN_THR": OPEN_THR,
                    "TAU_START": TAU_START, "TAU_END": TAU_END,
                    "N_TARGET": N_TARGET,
                    "EARLY_STOP_K": EARLY_STOP_K,
                }
            }, best_path)
            print(f"  >> Saved best to {best_path} "
                  f"(based on P@{EARLY_STOP_K}={val_metric_for_early_stop:.4f})")

    if os.path.exists(best_path):
        ckpt = torch.load(best_path, map_location=DEVICE)
        model = RankModelNoGate(text_hidden=TEXT_HIDDEN).to(DEVICE)
        model.load_state_dict(ckpt["model"])
        model.eval()
        print(f"\n[Info] Loaded best checkpoint from {best_path}")

    print("\n[Eval on TEST]")
    scores, labels_raw, codes, dates = infer_scores_full_day(model, test_loader, use_raw_labels=True)
    test_metrics = evaluate_daily_precision(scores, labels_raw, dates, top_list=EVAL_TOP_LIST, thr=OPEN_THR)
    test_ndcg    = evaluate_daily_ndcg(scores, labels_raw, dates, top_list=EVAL_TOP_LIST, thr=OPEN_THR)

    for k in EVAL_TOP_LIST:
        print(f"TEST P@{k}(y≥{OPEN_THR})={test_metrics[k]['Precision@K']:.4f} | "
              f"Avg@{k}={test_metrics[k]['AvgReturn@K']:.4f} | "
              f"NDCG@{k}={test_ndcg[k]['NDCG@K']:.4f}")

    export_daily_reco(
        scores, labels_raw, codes, dates,
        N=N_TARGET,
        path_csv="results/04_daily_only_features.csv"
    )


if __name__ == "__main__":
    train()

/tmp/ipykernel_4072/837733346.py:87: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_4072/837733346.py:88: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_4072/837733346.py:89: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider j

[Info] Inferred TEXT_HIDDEN=1024 from /root/autodl-tmp/telegram_emb_cache_clean/train_telegram_emb_filtered_filtered.npy (mmap)
[Run] Experiment 4: Daily only (Minute/Text zero, no gating)
[Epoch 1] TrainLoss=7.9732 | P@5(y≥0)=0.6185, Avg@5=0.1751, NDCG@5=0.6233
  >> Saved best to results/04_daily_only_features.pth (based on P@5=0.6185)
[Epoch 2] TrainLoss=7.9718 | P@5(y≥0)=0.5714, Avg@5=0.1732, NDCG@5=0.5735
[Epoch 3] TrainLoss=7.9716 | P@5(y≥0)=0.5244, Avg@5=0.1505, NDCG@5=0.5311
[Epoch 4] TrainLoss=7.9714 | P@5(y≥0)=0.5681, Avg@5=0.2179, NDCG@5=0.5737
[Epoch 5] TrainLoss=7.9713 | P@5(y≥0)=0.5345, Avg@5=0.1560, NDCG@5=0.5328
[Epoch 6] TrainLoss=7.9712 | P@5(y≥0)=0.5042, Avg@5=0.0353, NDCG@5=0.4871
[Epoch 7] TrainLoss=7.9712 | P@5(y≥0)=0.5513, Avg@5=0.2264, NDCG@5=0.5380
[Epoch 8] TrainLoss=7.9711 | P@5(y≥0)=0.5025, Avg@5=0.0560, NDCG@5=0.4922
[Epoch 9] TrainLoss=7.9710 | P@5(y≥0)=0.4958, Avg@5=-0.0929, NDCG@5=0.4907
[Epoch 10] TrainLoss=7.9710 | P@5(y≥0)=0.5042, Avg@5=-0.0423, NDCG@5

## Regression-Loss Variant

In [5]:
import os
import math
import random
from collections import defaultdict
from typing import List, Dict, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import _LRScheduler
from torch.utils.data import Dataset, DataLoader, Sampler


# ================== 全局与超参 ==================
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 数据列规格
DAILY_T, DAILY_F = 5, 43      # 5 步 × 43 维
MIN_T, MIN_F     = 5, 18      # 5 步 × 18 维
FEAT_TOTAL       = DAILY_T * DAILY_F + MIN_T * MIN_F

# embedding 目录（采用代码3的目录式管理）
TELEGRAM_EMB_DIR = "/root/autodl-tmp/telegram_emb_cache_clean"

# 训练批次采样（按“天”为组）
DAY_FORWARD_CHUNK = 4096

BATCH_WORKERS    = 4
PIN_MEMORY       = True

# 训练超参
EPOCHS           = 12
LR               = 1e-4
MIN_LR           = 3e-6
WEIGHT_DECAY     = 1e-4
MAX_GRAD_NORM    = 5.0
USE_AMP          = False
WARMUP_EPOCHS    = 2

# 模型宽度
LSTM_HIDDEN      = 128
ATTN_HIDDEN      = 64
MIN_CONV_C       = 128
FUSE_DIM         = 128
DROPOUT          = 0.22

# 业务目标：每天推荐 N 只（Top-K 的 K）
N_TARGET         = 20
OPEN_THR         = 0

# 评估：输出 Top@K 的命中与均值
EVAL_TOP_LIST    = [5, 10, 20]

# ========= 新增：早停/选模所依据的 K =========
EARLY_STOP_K     = 5


def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # torch.use_deterministic_algorithms(True)

seed_all()


# ================== 数据加载 ==================
TRAIN_CSV = "/root/autodl-tmp/train_filtered_filtered.csv"
VAL_CSV   = "/root/autodl-tmp/val_filtered_filtered.csv"
TEST_CSV  = "/root/autodl-tmp/test_filtered_filtered.csv"

train_df = pd.read_csv(TRAIN_CSV, low_memory=False)
val_df   = pd.read_csv(VAL_CSV,   low_memory=False)
test_df  = pd.read_csv(TEST_CSV,  low_memory=False)

# label clip（用于训练 loss）
p1  = train_df["opening_gains"].quantile(0.01)
p99 = train_df["opening_gains"].quantile(0.99)
train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
test_df["opening_gains_clipped"]  = test_df["opening_gains"].clip(lower=p1, upper=p99)

def cast_df(df: pd.DataFrame):
    feat_start = 2
    feat_end   = 2 + FEAT_TOTAL
    feats = df.iloc[:, feat_start:feat_end].astype(np.float32).values
    y_raw  = df["opening_gains"].astype(np.float32).values
    y_clip = df["opening_gains_clipped"].astype(np.float32).values
    code = df.iloc[:, 0].astype(str).values
    date = df.iloc[:, 1].astype(str).values
    return code, date, feats, y_clip, y_raw

tr_code, tr_date, tr_feats, tr_y, tr_y_raw = cast_df(train_df)
va_code, va_date, va_feats, va_y, va_y_raw = cast_df(val_df)
te_code, te_date, te_feats, te_y, te_y_raw = cast_df(test_df)

def build_groups(date_arr: np.ndarray) -> Dict[str, np.ndarray]:
    groups = defaultdict(list)
    for idx, d in enumerate(date_arr):
        groups[d].append(idx)
    return {k: np.array(v, dtype=np.int64) for k, v in groups.items()}

train_groups = build_groups(tr_date)
val_groups   = build_groups(va_date)
test_groups  = build_groups(te_date)

tr_dates = sorted(train_groups.keys())
va_dates = sorted(val_groups.keys())
te_dates = sorted(test_groups.keys())

tr_date2gid = {d: i for i, d in enumerate(tr_dates)}
va_date2gid = {d: i for i, d in enumerate(va_dates)}
te_date2gid = {d: i for i, d in enumerate(te_dates)}


# ================== embedding 读取 ==================
def load_embeddings(split: str, expected_n: int) -> np.ndarray:
    path = os.path.join(TELEGRAM_EMB_DIR, f"{split}_telegram_emb_filtered_filtered.npy")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Embedding file not found: {path}")
    emb = np.load(path).astype(np.float32)
    assert emb.ndim == 2, f"{split} emb must be 2D, got {emb.ndim}D"
    assert emb.shape[0] == expected_n, f"{split} emb rows mismatch: {emb.shape[0]} vs {expected_n}"
    print(f"[Info] Loaded {split} embeddings: shape={emb.shape} from {path}")
    return emb

tr_text = load_embeddings("train", tr_feats.shape[0])
va_text = load_embeddings("val",   va_feats.shape[0])
te_text = load_embeddings("test",  te_feats.shape[0])

TEXT_HIDDEN = tr_text.shape[1]
assert va_text.shape[1] == TEXT_HIDDEN and te_text.shape[1] == TEXT_HIDDEN, \
    f"text hidden mismatch across splits: train={TEXT_HIDDEN}, val={va_text.shape[1]}, test={te_text.shape[1]}"


# ================== Scheduler ==================
class WarmupCosine(_LRScheduler):
    def __init__(self, optimizer, warmup_epochs=1, max_epochs=EPOCHS, min_lr=1e-6, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.max_epochs = max_epochs
        self.min_lr = min_lr
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        epoch = self.last_epoch + 1
        lrs = []
        for base_lr in self.base_lrs:
            if epoch <= self.warmup_epochs:
                lr = base_lr * epoch / max(1, self.warmup_epochs)
            else:
                t = (epoch - self.warmup_epochs) / max(1, self.max_epochs - self.warmup_epochs)
                lr = self.min_lr + 0.5 * (base_lr - self.min_lr) * (1 + math.cos(math.pi * t))
            lrs.append(lr)
        return lrs


# ================== Dataset / Sampler / Collate ==================
class RankDataset(Dataset):
    def __init__(self, code, date, feats, y_clip, y_raw, text_emb: np.ndarray, date_to_gid: Dict[str, int]):
        self.code = code
        self.date = date
        self.feats = feats.astype(np.float32)
        self.y = y_clip.astype(np.float32)
        self.y_raw = y_raw.astype(np.float32)

        self.text_emb = text_emb.astype(np.float32)
        assert self.text_emb.shape[0] == self.feats.shape[0], "text_emb rows != feats rows"
        assert self.text_emb.shape[1] == TEXT_HIDDEN, "text_emb hidden != TEXT_HIDDEN"

        self.date_to_gid = date_to_gid
        self.gid = np.array([self.date_to_gid[d] for d in self.date], dtype=np.int64)

        self.daily_end = DAILY_T * DAILY_F
        self.min_end   = self.daily_end + MIN_T * MIN_F
        assert self.feats.shape[1] == self.min_end, f"feat dim mismatch: {self.feats.shape[1]} vs {self.min_end}"

    def __len__(self):
        return self.feats.shape[0]

    def __getitem__(self, idx):
        f = self.feats[idx]
        daily = f[:self.daily_end].reshape(DAILY_T, DAILY_F)
        minu  = f[self.daily_end:self.min_end].reshape(MIN_T, MIN_F)

        return {
            "daily": torch.from_numpy(daily),
            "minu":  torch.from_numpy(minu),
            "text":  torch.from_numpy(self.text_emb[idx]),
            "y":     torch.tensor(self.y[idx], dtype=torch.float32),
            "y_raw": torch.tensor(self.y_raw[idx], dtype=torch.float32),
            "gid":   torch.tensor(self.gid[idx], dtype=torch.long),
            "code":  self.code[idx],
            "date":  self.date[idx],
        }

class DayBatchSampler(Sampler[List[int]]):
    def __init__(self, groups: Dict[str, np.ndarray], shuffle=True, drop_last=False):
        self.groups = groups
        self.keys = list(groups.keys())
        self.shuffle = shuffle
        self.drop_last = drop_last

    def __iter__(self):
        keys = self.keys[:]
        if self.shuffle:
            random.shuffle(keys)
        for d in keys:
            idxs = self.groups[d]
            if idxs.size == 0:
                continue
            yield idxs.tolist()

    def __len__(self):
        return len(self.keys)

def collate_fn(batch):
    daily = torch.stack([b["daily"] for b in batch], dim=0)
    minu  = torch.stack([b["minu"]  for b in batch], dim=0)
    text  = torch.stack([b["text"]  for b in batch], dim=0)
    y     = torch.stack([b["y"]     for b in batch], dim=0)
    y_raw = torch.stack([b["y_raw"] for b in batch], dim=0)
    gid   = torch.stack([b["gid"]   for b in batch], dim=0)
    codes = [b["code"] for b in batch]
    dates = [b["date"] for b in batch]
    return daily, minu, text, y, y_raw, gid, codes, dates

train_set = RankDataset(tr_code, tr_date, tr_feats, tr_y, tr_y_raw, tr_text, tr_date2gid)
val_set   = RankDataset(va_code, va_date, va_feats, va_y, va_y_raw, va_text, va_date2gid)
test_set  = RankDataset(te_code, te_date, te_feats, te_y, te_y_raw, te_text, te_date2gid)

train_loader = DataLoader(
    train_set,
    batch_sampler=DayBatchSampler(train_groups, shuffle=True),
    num_workers=BATCH_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)
val_loader = DataLoader(
    val_set,
    batch_sampler=DayBatchSampler(val_groups, shuffle=False),
    num_workers=BATCH_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)
test_loader = DataLoader(
    test_set,
    batch_sampler=DayBatchSampler(test_groups, shuffle=False),
    num_workers=BATCH_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_fn
)


# ================== 模型（无门控版本） ==================
class AdditiveAttention(nn.Module):
    def __init__(self, in_dim, hidden_dim=ATTN_HIDDEN):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden_dim)
        self.v    = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, x, mask=None):
        h = torch.tanh(self.proj(x))
        scores = self.v(h).squeeze(-1)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        w = torch.softmax(scores, dim=-1)
        out = torch.bmm(w.unsqueeze(1), x).squeeze(1)
        return out, w

class DailyEncoder(nn.Module):
    def __init__(self, in_dim=DAILY_F, hidden=LSTM_HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=1, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.attn = AdditiveAttention(2 * hidden, ATTN_HIDDEN)

    def forward(self, x):
        h, _ = self.lstm(x)
        h = self.drop(h)
        out, _ = self.attn(h)
        return out

class MinuteEncoder(nn.Module):
    def __init__(self, in_dim=MIN_F, out_channels=MIN_CONV_C, dropout=DROPOUT):
        super().__init__()
        self.conv1 = nn.Conv1d(in_dim, out_channels, kernel_size=3, padding=1, dilation=1)
        self.bn1   = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=2, dilation=2)
        self.bn2   = nn.BatchNorm1d(out_channels)
        self.conv3 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=4, dilation=4)
        self.bn3   = nn.BatchNorm1d(out_channels)
        self.drop  = nn.Dropout(dropout)
        self.attn  = AdditiveAttention(out_channels, ATTN_HIDDEN)

    def forward(self, x):
        x = x.permute(0, 2, 1)
        h = F.relu(self.bn1(self.conv1(x)))
        h = F.relu(self.bn2(self.conv2(h)))
        h = F.relu(self.bn3(self.conv3(h)))
        h = self.drop(h)
        h = h.permute(0, 2, 1)
        out, _ = self.attn(h)
        return out

class RankModelNoGate(nn.Module):
    def __init__(self, text_hidden: int):
        super().__init__()
        self.daily_enc = DailyEncoder()
        self.min_enc   = MinuteEncoder()

        self.daily_proj = nn.Linear(2 * LSTM_HIDDEN, FUSE_DIM)
        self.min_proj   = nn.Linear(MIN_CONV_C,      FUSE_DIM)
        self.text_proj  = nn.Linear(text_hidden,     FUSE_DIM)

        self.fuse_proj = nn.Linear(3 * FUSE_DIM, FUSE_DIM)
        self.out = nn.Linear(FUSE_DIM, 1)

    def forward(self, daily_x, minu_x, text_x):
        d = self.daily_proj(self.daily_enc(daily_x))
        m = self.min_proj(self.min_enc(minu_x))
        t = self.text_proj(text_x)

        cat = torch.cat([d, m, t], dim=-1)
        fuse = F.relu(self.fuse_proj(cat))

        score = self.out(fuse).squeeze(-1)
        return score

def day_forward_scores(model, daily_x, minu_x, text_x, chunk: Optional[int]):
    B = daily_x.shape[0]
    if not chunk or chunk <= 0 or chunk >= B:
        return model(daily_x, minu_x, text_x)
    outs = []
    for st in range(0, B, chunk):
        ed = min(B, st + chunk)
        outs.append(model(daily_x[st:ed], minu_x[st:ed], text_x[st:ed]))
    return torch.cat(outs, dim=0)


# ================== 评估与导出 ==================
@torch.no_grad()
def infer_scores_full_day(model, loader, use_raw_labels=True):
    model.eval()
    all_scores, all_labels, all_codes, all_dates = [], [], [], []
    for daily_x, minu_x, text_x, y_clip, y_raw, gid, codes, dates in loader:
        daily_x = daily_x.to(DEVICE, non_blocking=True)
        minu_x  = minu_x.to(DEVICE, non_blocking=True)
        text_x  = text_x.to(DEVICE, non_blocking=True)

        s = day_forward_scores(model, daily_x, minu_x, text_x, DAY_FORWARD_CHUNK).detach().cpu().numpy()
        all_scores.append(s)

        lab = (y_raw if use_raw_labels else y_clip).numpy()
        all_labels.append(lab)
        all_codes.extend(codes)
        all_dates.extend(dates)

    return (np.concatenate(all_scores, axis=0),
            np.concatenate(all_labels, axis=0),
            all_codes, all_dates)

def evaluate_daily_precision(scores: np.ndarray, labels: np.ndarray, dates: list,
                             top_list=EVAL_TOP_LIST, thr=0):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    metrics = {k: {"prec": [], "mean_y": []} for k in top_list}
    cover_days = 0

    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        s_day = scores[idxs]
        y_day = labels[idxs]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        cover_days += 1
        for k in top_list:
            k_eff = min(k, s_day.size)
            topk = order[:k_eff]
            yk = y_day[topk]
            hit = (yk >= thr).astype(np.float32)
            metrics[k]["prec"].append(hit.mean() if k_eff > 0 else 0.0)
            metrics[k]["mean_y"].append(yk.mean() if k_eff > 0 else 0.0)

    out = {}
    for k in top_list:
        out[k] = {
            "Precision@K": float(np.mean(metrics[k]["prec"])) if metrics[k]["prec"] else 0.0,
            "AvgReturn@K": float(np.mean(metrics[k]["mean_y"])) if metrics[k]["mean_y"] else 0.0,
            "CoveredDays": int(cover_days),
        }
    return out

def evaluate_daily_ndcg(scores: np.ndarray, labels: np.ndarray, dates: list,
                        top_list=EVAL_TOP_LIST, thr=OPEN_THR):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    def dcg_at_k(rels_sorted_by_rank, k):
        k_eff = min(k, len(rels_sorted_by_rank))
        if k_eff <= 0:
            return 0.0
        gains = (2 ** rels_sorted_by_rank[:k_eff] - 1.0)
        discounts = 1.0 / np.log2(np.arange(2, 2 + k_eff))
        return float(np.sum(gains * discounts))

    bucket = {k: [] for k in top_list}

    for d, idxs in idx_by_day.items():
        arr = np.array(idxs, dtype=np.int64)
        s_day = scores[arr]
        y_day = labels[arr]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        rel = (y_day >= thr).astype(np.int32)
        rel_sorted_gt = np.sort(rel)[::-1]

        for k in top_list:
            k_eff = min(k, s_day.size)
            if k_eff <= 0:
                bucket[k].append(0.0)
                continue
            rel_ranked = rel[order]
            dcg = dcg_at_k(rel_ranked, k_eff)
            idcg = dcg_at_k(rel_sorted_gt, k_eff)
            ndcg = (dcg / idcg) if idcg > 0 else 0.0
            bucket[k].append(ndcg)

    out = {}
    for k in top_list:
        vals = bucket[k]
        out[k] = {
            "NDCG@K": float(np.mean(vals)) if vals else 0.0,
            "CoveredDays": len(vals),
        }
    return out

def export_daily_reco(scores: np.ndarray, labels_raw: np.ndarray, codes: list, dates: list,
                      N=N_TARGET, path_csv="daily_recommendations.csv"):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    rows = []
    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        order = np.argsort(-scores[idxs])
        topn = order[:min(N, idxs.size)]
        for r, j in enumerate(topn, start=1):
            i = idxs[j]
            rows.append({
                "date": dates[i],
                "rank": r,
                "code": codes[i],
                "score": float(scores[i]),
                "true_open_return": float(labels_raw[i])
            })
    pd.DataFrame(rows).sort_values(["date", "rank"]).to_csv(path_csv, index=False)
    print(f"[Export] Saved daily recommendations to {path_csv}")


# ================== Train（仅回归损失，无门控） ==================
seed_all(SEED)
def train():
    model = RankModelNoGate(text_hidden=TEXT_HIDDEN).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler(DEVICE, enabled=(USE_AMP and DEVICE == "cuda"))
    scheduler = WarmupCosine(optimizer, warmup_epochs=WARMUP_EPOCHS, max_epochs=EPOCHS, min_lr=MIN_LR)

    best_metric = -np.inf
    best_path = "results/05_regression_Loss_variant.pth"
    print("[Training Mode] Using Regression Loss (MSE only, No Gating)")

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss, days = 0.0, 0

        for daily_x, minu_x, text_x, y_clip, y_raw, gid, codes, dates in train_loader:
            daily_x = daily_x.to(DEVICE, non_blocking=True)
            minu_x  = minu_x.to(DEVICE, non_blocking=True)
            text_x  = text_x.to(DEVICE, non_blocking=True)
            y_clip  = y_clip.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if USE_AMP and DEVICE == "cuda":
                with torch.amp.autocast(DEVICE):
                    s = day_forward_scores(model, daily_x, minu_x, text_x, DAY_FORWARD_CHUNK)
                    loss = F.mse_loss(s, y_clip)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                s = day_forward_scores(model, daily_x, minu_x, text_x, DAY_FORWARD_CHUNK)
                loss = F.mse_loss(s, y_clip)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                optimizer.step()

            total_loss += float(loss.item())
            days += 1

        scheduler.step()
        tr_loss = total_loss / max(1, days)

        scores, labels_raw, codes, dates = infer_scores_full_day(model, val_loader, use_raw_labels=True)

        val_metrics_main = evaluate_daily_precision(
            scores, labels_raw, dates, top_list=[EARLY_STOP_K], thr=OPEN_THR
        )
        val_ndcg_main = evaluate_daily_ndcg(
            scores, labels_raw, dates, top_list=[EARLY_STOP_K], thr=OPEN_THR
        )

        p_main   = val_metrics_main[EARLY_STOP_K]["Precision@K"]
        avg_main = val_metrics_main[EARLY_STOP_K]["AvgReturn@K"]
        ndcg_main = val_ndcg_main[EARLY_STOP_K]["NDCG@K"]

        val_metric_for_early_stop = p_main

        print(f"[Epoch {epoch}] TrainLoss={tr_loss:.4f} | "
              f"P@{EARLY_STOP_K}(y≥{OPEN_THR})={p_main:.4f}, "
              f"Avg@{EARLY_STOP_K}={avg_main:.4f}, "
              f"NDCG@{EARLY_STOP_K}={ndcg_main:.4f}")

        if val_metric_for_early_stop > best_metric:
            best_metric = val_metric_for_early_stop
            torch.save({
                "model": model.state_dict(),
                "cfg": {
                    "DAILY_T": DAILY_T, "DAILY_F": DAILY_F,
                    "MIN_T": MIN_T, "MIN_F": MIN_F,
                    "TEXT_HIDDEN": TEXT_HIDDEN,
                    "OPEN_THR": OPEN_THR,
                    "N_TARGET": N_TARGET,
                    "EARLY_STOP_K": EARLY_STOP_K,
                }
            }, best_path)
            print(f"  >> Saved best to {best_path} "
                  f"(based on P@{EARLY_STOP_K}={val_metric_for_early_stop:.4f})")

    if os.path.exists(best_path):
        ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
        model = RankModelNoGate(text_hidden=TEXT_HIDDEN).to(DEVICE)
        model.load_state_dict(ckpt["model"])
        model.eval()
        print(f"\n[Info] Loaded best checkpoint from {best_path}")

    print("\n[Eval on TEST]")
    scores, labels_raw, codes, dates = infer_scores_full_day(model, test_loader, use_raw_labels=True)
    test_metrics = evaluate_daily_precision(scores, labels_raw, dates, top_list=EVAL_TOP_LIST, thr=OPEN_THR)
    test_ndcg    = evaluate_daily_ndcg(scores, labels_raw, dates, top_list=EVAL_TOP_LIST, thr=OPEN_THR)

    for k in EVAL_TOP_LIST:
        print(f"TEST P@{k}(y≥{OPEN_THR})={test_metrics[k]['Precision@K']:.4f} | "
              f"Avg@{k}={test_metrics[k]['AvgReturn@K']:.4f} | "
              f"NDCG@{k}={test_ndcg[k]['NDCG@K']:.4f} | "
              f"Days={test_metrics[k]['CoveredDays']}")

    export_daily_reco(
        scores, labels_raw, codes, dates,
        N=N_TARGET,
        path_csv="results/05_regression_Loss_variant.csv"
    )


if __name__ == "__main__":
    train()

/tmp/ipykernel_4072/3677505916.py:87: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_4072/3677505916.py:88: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_4072/3677505916.py:89: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Conside

[Info] Loaded train embeddings: shape=(1610196, 1024) from /root/autodl-tmp/telegram_emb_cache_clean/train_telegram_emb_filtered_filtered.npy
[Info] Loaded val embeddings: shape=(349618, 1024) from /root/autodl-tmp/telegram_emb_cache_clean/val_telegram_emb_filtered_filtered.npy
[Info] Loaded test embeddings: shape=(604975, 1024) from /root/autodl-tmp/telegram_emb_cache_clean/test_telegram_emb_filtered_filtered.npy
[Training Mode] Using Regression Loss (MSE only, No Gating)
[Epoch 1] TrainLoss=0.6646 | P@5(y≥0)=0.5714, Avg@5=0.0679, NDCG@5=0.5734
  >> Saved best to results/05_regression_Loss_variant.pth (based on P@5=0.5714)
[Epoch 2] TrainLoss=0.6523 | P@5(y≥0)=0.5479, Avg@5=0.0672, NDCG@5=0.5469
[Epoch 3] TrainLoss=0.6423 | P@5(y≥0)=0.5950, Avg@5=0.1892, NDCG@5=0.5922
  >> Saved best to results/05_regression_Loss_variant.pth (based on P@5=0.5950)
[Epoch 4] TrainLoss=0.6364 | P@5(y≥0)=0.5798, Avg@5=0.1621, NDCG@5=0.5707
[Epoch 5] TrainLoss=0.6336 | P@5(y≥0)=0.5933, Avg@5=0.1914, NDCG@5

# XGBoost

In [1]:
import os
import random
from collections import defaultdict
from typing import Dict

import numpy as np
import pandas as pd
import xgboost as xgb

# ================== 全局严格复现设置 ==================
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

random.seed(SEED)
np.random.seed(SEED)

# ================== 路径与参数 ==================
TRAIN_CSV = "/root/autodl-tmp/train_filtered_filtered.csv"
VAL_CSV   = "/root/autodl-tmp/val_filtered_filtered.csv"
TEST_CSV  = "/root/autodl-tmp/test_filtered_filtered.csv"

TELEGRAM_EMB_DIR = "/root/autodl-tmp/telegram_emb_cache_clean"

# 特征维度（与原程序一致）
DAILY_T, DAILY_F = 5, 43
MIN_T, MIN_F     = 5, 18
FEAT_TOTAL       = DAILY_T * DAILY_F + MIN_T * MIN_F

# 业务目标
OPEN_THR = 0.0
N_TARGET = 20
EVAL_TOP_LIST = [5, 10, 20]

# 训练参数
NUM_BOOST_ROUND = 1000
EARLY_STOPPING_ROUNDS = 50

SAVE_RECO_PATH  = "results/06_xgboost.csv"
SAVE_MODEL_PATH = "results/06_xgboost.model"
SAVE_META_PATH  = "results/06_xgboost_meta.txt"

# ================== XGBoost CPU 严格复现参数 ==================
XGB_PARAMS = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",

    "max_depth": 8,
    "learning_rate": 0.05,

    # 关键：关闭随机采样
    "subsample": 1.0,
    "colsample_bytree": 1.0,

    # 正则项
    "min_child_weight": 1.0,
    "lambda": 1.0,
    "alpha": 0.0,

    # CPU + 单线程，优先保证复现
    "tree_method": "hist",
    "nthread": 1,

    # 随机种子
    "seed": SEED,
    "random_state": SEED,
}

# ================== 工具函数 ==================
def ensure_parent_dir(path: str):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)

def load_embeddings(split: str, expected_n: int) -> np.ndarray:
    path = os.path.join(TELEGRAM_EMB_DIR, f"{split}_telegram_emb_filtered_filtered.npy")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Embedding file not found: {path}")

    emb = np.load(path).astype(np.float32)
    if emb.shape[0] != expected_n:
        raise ValueError(f"{split} emb rows mismatch: expected {expected_n}, got {emb.shape[0]}")

    print(f"[Info] Loaded {split} embeddings: shape={emb.shape}")
    return emb

def stable_sort_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    固定输入顺序，避免样本顺序波动影响训练路径
    默认第0列为 code，第1列为 date，与原代码保持一致
    """
    code_col = df.columns[0]
    date_col = df.columns[1]
    return df.sort_values([date_col, code_col], kind="mergesort").reset_index(drop=True)

def cast_df(df: pd.DataFrame, feat_start=2, feat_end=2 + FEAT_TOTAL):
    feats = df.iloc[:, feat_start:feat_end].astype(np.float32).values
    y_raw = df["opening_gains"].astype(np.float32).values
    code = df.iloc[:, 0].astype(str).values
    date = df.iloc[:, 1].astype(str).values
    return code, date, feats, y_raw

# ================== 评估函数 ==================
def evaluate_daily_precision(scores: np.ndarray, labels: np.ndarray, dates,
                             top_list=EVAL_TOP_LIST, thr=0.0):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    metrics = {k: {"prec": [], "mean_y": []} for k in top_list}
    cover_days = 0

    for d, idxs in idx_by_day.items():
        idxs = np.asarray(idxs, dtype=np.int64)
        s_day = scores[idxs]
        y_day = labels[idxs]
        if s_day.size == 0:
            continue

        order = np.argsort(-s_day, kind="mergesort")
        cover_days += 1

        for k in top_list:
            k_eff = min(k, s_day.size)
            topk = order[:k_eff]
            yk = y_day[topk]
            hit = (yk >= thr).astype(np.float32)

            metrics[k]["prec"].append(hit.mean() if k_eff > 0 else 0.0)
            metrics[k]["mean_y"].append(yk.mean() if k_eff > 0 else 0.0)

    out = {}
    for k in top_list:
        out[k] = {
            "Precision@K": float(np.mean(metrics[k]["prec"])) if metrics[k]["prec"] else 0.0,
            "AvgReturn@K": float(np.mean(metrics[k]["mean_y"])) if metrics[k]["mean_y"] else 0.0,
            "CoveredDays": int(cover_days),
        }
    return out

def evaluate_daily_ndcg(scores: np.ndarray, labels: np.ndarray, dates,
                        top_list=EVAL_TOP_LIST, thr=OPEN_THR):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    def dcg_at_k(rels_sorted_by_rank, k):
        k_eff = min(k, len(rels_sorted_by_rank))
        if k_eff <= 0:
            return 0.0
        gains = (2 ** rels_sorted_by_rank[:k_eff] - 1.0)
        discounts = 1.0 / np.log2(np.arange(2, 2 + k_eff))
        return float(np.sum(gains * discounts))

    bucket = {k: [] for k in top_list}

    for d, idxs in idx_by_day.items():
        arr = np.asarray(idxs, dtype=np.int64)
        s_day = scores[arr]
        y_day = labels[arr]
        if s_day.size == 0:
            continue

        order = np.argsort(-s_day, kind="mergesort")
        rel = (y_day >= thr).astype(np.int32)
        rel_sorted_gt = np.sort(rel)[::-1]

        for k in top_list:
            k_eff = min(k, s_day.size)
            if k_eff <= 0:
                bucket[k].append(0.0)
                continue

            rel_ranked = rel[order]
            dcg = dcg_at_k(rel_ranked, k_eff)
            idcg = dcg_at_k(rel_sorted_gt, k_eff)
            ndcg = (dcg / idcg) if idcg > 0 else 0.0
            bucket[k].append(ndcg)

    out = {}
    for k in top_list:
        vals = bucket[k]
        out[k] = {
            "NDCG@K": float(np.mean(vals)) if vals else 0.0,
            "CoveredDays": len(vals),
        }
    return out

def export_daily_reco(scores: np.ndarray, labels_raw: np.ndarray, codes, dates,
                      N=N_TARGET, path_csv=SAVE_RECO_PATH):
    ensure_parent_dir(path_csv)

    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    rows = []
    for d, idxs in idx_by_day.items():
        idxs = np.asarray(idxs, dtype=np.int64)
        order = np.argsort(-scores[idxs], kind="mergesort")
        topn = order[:min(N, idxs.size)]

        for r, j in enumerate(topn, start=1):
            i = idxs[j]
            rows.append({
                "date": dates[i],
                "rank": r,
                "code": codes[i],
                "score": float(scores[i]),
                "true_open_return": float(labels_raw[i]),
            })

    pd.DataFrame(rows).sort_values(["date", "rank"], kind="mergesort").to_csv(path_csv, index=False)
    print(f"[Export] Saved recommendations to: {path_csv}")

# ================== 主流程 ==================
def main():
    print("=" * 80)
    print("XGBoost CPU Strict Reproducible Version")
    print(f"xgboost version: {xgb.__version__}")
    print(f"SEED: {SEED}")
    print("=" * 80)

    ensure_parent_dir(SAVE_MODEL_PATH)
    ensure_parent_dir(SAVE_RECO_PATH)
    ensure_parent_dir(SAVE_META_PATH)

    # ---------- 读取数据 ----------
    train_df = pd.read_csv(TRAIN_CSV, low_memory=False)
    val_df   = pd.read_csv(VAL_CSV, low_memory=False)
    test_df  = pd.read_csv(TEST_CSV, low_memory=False)

    # ---------- 固定顺序 ----------
    train_df = stable_sort_df(train_df)
    val_df   = stable_sort_df(val_df)
    test_df  = stable_sort_df(test_df)

    print(f"[Info] train shape = {train_df.shape}")
    print(f"[Info] val   shape = {val_df.shape}")
    print(f"[Info] test  shape = {test_df.shape}")

    # ---------- 标签裁剪（与原程序一致） ----------
    p1  = train_df["opening_gains"].quantile(0.01)
    p99 = train_df["opening_gains"].quantile(0.99)

    train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
    val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
    test_df["opening_gains_clipped"]  = test_df["opening_gains"].clip(lower=p1, upper=p99)

    print(f"[Info] clip range = [{p1:.6f}, {p99:.6f}]")

    # ---------- 拆分 ----------
    tr_code, tr_date, tr_feats, tr_y_raw = cast_df(train_df)
    va_code, va_date, va_feats, va_y_raw = cast_df(val_df)
    te_code, te_date, te_feats, te_y_raw = cast_df(test_df)

    tr_y = train_df["opening_gains_clipped"].astype(np.float32).values
    va_y = val_df["opening_gains_clipped"].astype(np.float32).values
    te_y = test_df["opening_gains_clipped"].astype(np.float32).values

    # ---------- 文本嵌入 ----------
    tr_text = load_embeddings("train", tr_feats.shape[0])
    va_text = load_embeddings("val",   va_feats.shape[0])
    te_text = load_embeddings("test",  te_feats.shape[0])

    # ---------- 拼接特征 ----------
    X_train = np.concatenate([tr_feats, tr_text], axis=1).astype(np.float32, copy=False)
    X_val   = np.concatenate([va_feats, va_text], axis=1).astype(np.float32, copy=False)
    X_test  = np.concatenate([te_feats, te_text], axis=1).astype(np.float32, copy=False)

    print(f"[Info] Final feature dim: {X_train.shape[1]} (num={FEAT_TOTAL}, text={tr_text.shape[1]})")
    print(f"[Info] X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")

    # ---------- DMatrix ----------
    dtrain = xgb.DMatrix(X_train, label=tr_y)
    dval   = xgb.DMatrix(X_val,   label=va_y)
    dtest  = xgb.DMatrix(X_test,  label=te_y)

    # ---------- 训练 ----------
    print("\n[Training XGBoost on CPU...]")
    evals_result = {}

    model = xgb.train(
        params=XGB_PARAMS,
        dtrain=dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dtrain, "train"), (dval, "val")],
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        evals_result=evals_result,
        verbose_eval=100,
    )

    best_iteration = model.best_iteration
    best_score = model.best_score

    print(f"\n[Info] Best iteration = {best_iteration}")
    print(f"[Info] Best val rmse   = {best_score}")

    # ---------- 预测 ----------
    pred_val = model.predict(dval, iteration_range=(0, best_iteration + 1))
    pred_test = model.predict(dtest, iteration_range=(0, best_iteration + 1))

    # ---------- RMSE ----------
    val_rmse = float(np.sqrt(np.mean((pred_val - va_y) ** 2)))
    test_rmse = float(np.sqrt(np.mean((pred_test - te_y) ** 2)))

    print(f"[Info] VAL  RMSE(clipped)  = {val_rmse:.6f}")
    print(f"[Info] TEST RMSE(clipped)  = {test_rmse:.6f}")

    # ---------- 测试集评估（原始标签） ----------
    print("\n[Evaluation on TEST set]")
    test_metrics = evaluate_daily_precision(
        pred_test, te_y_raw, te_date, top_list=EVAL_TOP_LIST, thr=OPEN_THR
    )
    test_ndcg = evaluate_daily_ndcg(
        pred_test, te_y_raw, te_date, top_list=EVAL_TOP_LIST, thr=OPEN_THR
    )

    for k in EVAL_TOP_LIST:
        print(
            f"XGBoost TEST P@{k}={test_metrics[k]['Precision@K']:.4f} | "
            f"Avg@{k}={test_metrics[k]['AvgReturn@K']:.4f} | "
            f"NDCG@{k}={test_ndcg[k]['NDCG@K']:.4f} | "
            f"Days={test_metrics[k]['CoveredDays']}"
        )

    # ---------- 导出推荐 ----------
    export_daily_reco(
        pred_test,
        te_y_raw,
        te_code,
        te_date,
        N=N_TARGET,
        path_csv=SAVE_RECO_PATH
    )

    # ---------- 保存模型 ----------
    model.save_model(SAVE_MODEL_PATH)
    print(f"[Info] Model saved to: {SAVE_MODEL_PATH}")

    # ---------- 保存元信息 ----------
    with open(SAVE_META_PATH, "w", encoding="utf-8") as f:
        f.write(f"xgboost_version={xgb.__version__}\n")
        f.write(f"seed={SEED}\n")
        f.write(f"best_iteration={best_iteration}\n")
        f.write(f"best_score={best_score}\n")
        f.write(f"num_boost_round={NUM_BOOST_ROUND}\n")
        f.write(f"early_stopping_rounds={EARLY_STOPPING_ROUNDS}\n")
        for k, v in XGB_PARAMS.items():
            f.write(f"{k}={v}\n")

    print(f"[Info] Meta saved to: {SAVE_META_PATH}")

if __name__ == "__main__":
    main()

XGBoost CPU Strict Reproducible Version
xgboost version: 3.2.0
SEED: 42
[Info] train shape = (1610196, 309)
[Info] val   shape = (349618, 309)
[Info] test  shape = (604975, 309)
[Info] clip range = [-3.235294, 2.793296]


/tmp/ipykernel_1374/3063328109.py:250: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_1374/3063328109.py:251: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_1374/3063328109.py:252: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

[Info] Loaded train embeddings: shape=(1610196, 1024)
[Info] Loaded val embeddings: shape=(349618, 1024)
[Info] Loaded test embeddings: shape=(604975, 1024)
[Info] Final feature dim: 1329 (num=305, text=1024)
[Info] X_train=(1610196, 1329), X_val=(349618, 1329), X_test=(604975, 1329)

[Training XGBoost on CPU...]
[0]	train-rmse:0.81877	val-rmse:0.94918
[100]	train-rmse:0.76395	val-rmse:0.94258
[200]	train-rmse:0.74749	val-rmse:0.94015
[300]	train-rmse:0.73682	val-rmse:0.93952
[390]	train-rmse:0.72859	val-rmse:0.93946

[Info] Best iteration = 340
[Info] Best val rmse   = 0.9392295166578404
[Info] VAL  RMSE(clipped)  = 0.939229
[Info] TEST RMSE(clipped)  = 0.827765

[Evaluation on TEST set]
XGBoost TEST P@5=0.6517 | Avg@5=0.3389 | NDCG@5=0.6718 | Days=205
XGBoost TEST P@10=0.6400 | Avg@10=0.1862 | NDCG@10=0.6568 | Days=205
XGBoost TEST P@20=0.6407 | Avg@20=0.1104 | NDCG@20=0.6513 | Days=205
[Export] Saved recommendations to: results/06_xgboost.csv
[Info] Model saved to: results/06_xgboos

/tmp/ipykernel_1374/3063328109.py:342: UserWarning: [11:18:27] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1573: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  model.save_model(SAVE_MODEL_PATH)


# LightGBM

In [1]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import Dict, List

import lightgbm as lgb

# ================== 路径与参数 ==================
TRAIN_CSV = "/root/autodl-tmp/train_filtered_filtered.csv"
VAL_CSV   = "/root/autodl-tmp/val_filtered_filtered.csv"
TEST_CSV  = "/root/autodl-tmp/test_filtered_filtered.csv"

TELEGRAM_EMB_DIR = "/root/autodl-tmp/telegram_emb_cache_clean"

# 特征维度（必须与主模型一致）
DAILY_T, DAILY_F = 5, 43
MIN_T, MIN_F     = 5, 18
FEAT_TOTAL       = DAILY_T * DAILY_F + MIN_T * MIN_F

# 业务目标
OPEN_THR = 0.0
N_TARGET = 20
EVAL_TOP_LIST = [5, 10, 20]

# LightGBM 超参（可调）
LGB_PARAMS = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 63,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 1,
    'max_depth': -1,
    'min_data_in_leaf': 20,
    'verbosity': -1,
    'random_state': 42,
    # 启用 GPU（如果可用且已安装 GPU 版 LightGBM）
    # 'device': 'gpu',
    # 'gpu_platform_id': 0,
    # 'gpu_device_id': 0,
}

NUM_BOOST_ROUND = 2000
EARLY_STOPPING_ROUNDS = 50


# ================== 工具函数 ==================
def load_embeddings(split: str, expected_n: int) -> np.ndarray:
    path = os.path.join(TELEGRAM_EMB_DIR, f"{split}_telegram_emb_filtered_filtered.npy")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Embedding file not found: {path}")
    emb = np.load(path).astype(np.float32)
    assert emb.shape[0] == expected_n, f"{split} emb rows mismatch"
    print(f"[Info] Loaded {split} embeddings: shape={emb.shape}")
    return emb

def cast_df(df: pd.DataFrame, feat_start=2, feat_end=2+FEAT_TOTAL):
    feats = df.iloc[:, feat_start:feat_end].astype(np.float32).values
    y_raw = df["opening_gains"].astype(np.float32).values
    code = df.iloc[:, 0].astype(str).values
    date = df.iloc[:, 1].astype(str).values
    return code, date, feats, y_raw

# ================== 评估函数（复用逻辑） ==================
def evaluate_daily_precision(scores: np.ndarray, labels: np.ndarray, dates: list,
                             top_list=EVAL_TOP_LIST, thr=0):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    metrics = {k: {"prec": [], "mean_y": []} for k in top_list}
    cover_days = 0

    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        s_day = scores[idxs]
        y_day = labels[idxs]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        cover_days += 1
        for k in top_list:
            k_eff = min(k, s_day.size)
            topk = order[:k_eff]
            yk = y_day[topk]
            hit = (yk >= thr).astype(np.float32)
            metrics[k]["prec"].append(hit.mean() if k_eff > 0 else 0.0)
            metrics[k]["mean_y"].append(yk.mean() if k_eff > 0 else 0.0)

    out = {}
    for k in top_list:
        out[k] = {
            "Precision@K": float(np.mean(metrics[k]["prec"])) if metrics[k]["prec"] else 0.0,
            "AvgReturn@K": float(np.mean(metrics[k]["mean_y"])) if metrics[k]["mean_y"] else 0.0,
            "CoveredDays": int(cover_days),
        }
    return out

def evaluate_daily_ndcg(scores: np.ndarray, labels: np.ndarray, dates: list,
                        top_list=EVAL_TOP_LIST, thr=OPEN_THR):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    def dcg_at_k(rels_sorted_by_rank, k):
        k_eff = min(k, len(rels_sorted_by_rank))
        if k_eff <= 0:
            return 0.0
        gains = (2 ** rels_sorted_by_rank[:k_eff] - 1.0)
        discounts = 1.0 / np.log2(np.arange(2, 2 + k_eff))
        return float(np.sum(gains * discounts))

    bucket = {k: [] for k in top_list}

    for d, idxs in idx_by_day.items():
        arr = np.array(idxs, dtype=np.int64)
        s_day = scores[arr]
        y_day = labels[arr]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        rel = (y_day >= thr).astype(np.int32)
        rel_sorted_gt = np.sort(rel)[::-1]

        for k in top_list:
            k_eff = min(k, s_day.size)
            if k_eff <= 0:
                bucket[k].append(0.0)
                continue
            rel_ranked = rel[order]
            dcg = dcg_at_k(rel_ranked, k_eff)
            idcg = dcg_at_k(rel_sorted_gt, k_eff)
            ndcg = (dcg / idcg) if idcg > 0 else 0.0
            bucket[k].append(ndcg)

    out = {}
    for k in top_list:
        vals = bucket[k]
        out[k] = {
            "NDCG@K": float(np.mean(vals)) if vals else 0.0,
            "CoveredDays": len(vals),
        }
    return out

def export_daily_reco(scores: np.ndarray, labels_raw: np.ndarray, codes: list, dates: list,
                      N=N_TARGET, path_csv="lgb_recommendations.csv"):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    rows = []
    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        order = np.argsort(-scores[idxs])
        topn = order[:min(N, idxs.size)]
        for r, j in enumerate(topn, start=1):
            i = idxs[j]
            rows.append({
                "date": dates[i],
                "rank": r,
                "code": codes[i],
                "score": float(scores[i]),
                "true_open_return": float(labels_raw[i])
            })
    pd.DataFrame(rows).sort_values(["date", "rank"]).to_csv(path_csv, index=False)
    print(f"[Export] Saved LightGBM recommendations to {path_csv}")


# ================== 主流程 ==================
def main():
    # --- 加载数据 ---
    train_df = pd.read_csv(TRAIN_CSV, low_memory=False)
    val_df   = pd.read_csv(VAL_CSV,   low_memory=False)
    test_df  = pd.read_csv(TEST_CSV,  low_memory=False)

    # 标签裁剪（与主模型一致）
    p1  = train_df["opening_gains"].quantile(0.01)
    p99 = train_df["opening_gains"].quantile(0.99)
    train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
    val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
    test_df["opening_gains_clipped"]  = test_df["opening_gains"].clip(lower=p1, upper=p99)

    tr_code, tr_date, tr_feats, tr_y_raw = cast_df(train_df)
    va_code, va_date, va_feats, va_y_raw = cast_df(val_df)
    te_code, te_date, te_feats, te_y_raw = cast_df(test_df)

    tr_y = train_df["opening_gains_clipped"].values.astype(np.float32)
    va_y = val_df["opening_gains_clipped"].values.astype(np.float32)
    te_y = test_df["opening_gains_clipped"].values.astype(np.float32)

    # --- 加载文本嵌入 ---
    tr_text = load_embeddings("train", tr_feats.shape[0])
    va_text = load_embeddings("val",   va_feats.shape[0])
    te_text = load_embeddings("test",  te_feats.shape[0])

    # --- 拼接特征 ---
    X_train = np.concatenate([tr_feats, tr_text], axis=1)
    X_val   = np.concatenate([va_feats, va_text], axis=1)
    X_test  = np.concatenate([te_feats, te_text], axis=1)

    print(f"Final feature dim: {X_train.shape[1]} (num={FEAT_TOTAL} + text={tr_text.shape[1]})")

    # --- 构建 LightGBM Dataset（支持 group/group weight）---
    train_data = lgb.Dataset(X_train, label=tr_y, free_raw_data=False)
    val_data   = lgb.Dataset(X_val,   label=va_y, reference=train_data)

    # --- 训练 ---
    print("\n[Training LightGBM...]")
    model = lgb.train(
        LGB_PARAMS,
        train_data,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'val'],
        num_boost_round=NUM_BOOST_ROUND,
        callbacks=[
            lgb.log_evaluation(100),
            lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=True)
        ]
    )

    # --- 预测 ---
    pred_val = model.predict(X_val, num_iteration=model.best_iteration)
    pred_test = model.predict(X_test, num_iteration=model.best_iteration)

    # --- 评估（使用原始标签）---
    print("\n[Evaluation on TEST set]")
    test_metrics = evaluate_daily_precision(pred_test, te_y_raw, te_date, top_list=EVAL_TOP_LIST, thr=OPEN_THR)
    test_ndcg    = evaluate_daily_ndcg(pred_test, te_y_raw, te_date, top_list=EVAL_TOP_LIST, thr=OPEN_THR)

    for k in EVAL_TOP_LIST:
        print(f"LightGBM TEST P@{k}={test_metrics[k]['Precision@K']:.4f} | "
              f"Avg@{k}={test_metrics[k]['AvgReturn@K']:.4f} | "
              f"NDCG@{k}={test_ndcg[k]['NDCG@K']:.4f} | "
              f"Days={test_metrics[k]['CoveredDays']}")

    # --- 导出推荐 ---
    export_daily_reco(pred_test, te_y_raw, te_code, te_date, N=N_TARGET, path_csv="results/07_lightgbm.csv")

    # --- 可选：保存模型 ---
    model.save_model("results/07_lightgbm.txt")
    print("[Info] LightGBM model saved to lightgbm_baseline.txt")


if __name__ == "__main__":
    main()

/tmp/ipykernel_9273/634150154.py:182: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_9273/634150154.py:183: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_9273/634150154.py:184: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Conside

[Info] Loaded train embeddings: shape=(1610196, 1024)
[Info] Loaded val embeddings: shape=(349618, 1024)
[Info] Loaded test embeddings: shape=(604975, 1024)
Final feature dim: 1329 (num=305 + text=1024)

[Training LightGBM...]
Training until validation scores don't improve for 50 rounds
[100]	train's rmse: 0.775556	val's rmse: 0.943754
[200]	train's rmse: 0.76417	val's rmse: 0.940893
[300]	train's rmse: 0.75702	val's rmse: 0.938765
[400]	train's rmse: 0.751744	val's rmse: 0.938003
[500]	train's rmse: 0.746959	val's rmse: 0.937458
Early stopping, best iteration is:
[542]	train's rmse: 0.745107	val's rmse: 0.937104

[Evaluation on TEST set]
LightGBM TEST P@5=0.6595 | Avg@5=0.3540 | NDCG@5=0.6766 | Days=205
LightGBM TEST P@10=0.6566 | Avg@10=0.2060 | NDCG@10=0.6685 | Days=205
LightGBM TEST P@20=0.6402 | Avg@20=0.1102 | NDCG@20=0.6525 | Days=205
[Export] Saved LightGBM recommendations to results/07_lightgbm.csv
[Info] LightGBM model saved to lightgbm_baseline.txt


# MLP

In [1]:
import os
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import Dict, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

# ================== 路径与参数 ==================
TRAIN_CSV = "/root/autodl-tmp/train_filtered_filtered.csv"
VAL_CSV   = "/root/autodl-tmp/val_filtered_filtered.csv"
TEST_CSV  = "/root/autodl-tmp/test_filtered_filtered.csv"

TELEGRAM_EMB_DIR = "/root/autodl-tmp/telegram_emb_cache_clean"

# 特征维度（必须与主模型一致）
DAILY_T, DAILY_F = 5, 43
MIN_T, MIN_F     = 5, 18
FEAT_TOTAL       = DAILY_T * DAILY_F + MIN_T * MIN_F

# 业务目标
OPEN_THR = 0.0
N_TARGET = 20
EVAL_TOP_LIST = [5, 10, 20]

# MLP 超参
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 2048
LR = 1e-3
WEIGHT_DECAY = 1e-5
EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
DROP_RATE = 0.3

# 网络结构
HIDDEN_SIZES = [512, 256, 128]  # 可调


# ================== 工具函数 ==================
def load_embeddings(split: str, expected_n: int) -> np.ndarray:
    path = os.path.join(TELEGRAM_EMB_DIR, f"{split}_telegram_emb_filtered_filtered.npy")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Embedding file not found: {path}")
    emb = np.load(path).astype(np.float32)
    assert emb.shape[0] == expected_n, f"{split} emb rows mismatch"
    print(f"[Info] Loaded {split} embeddings: shape={emb.shape}")
    return emb

def cast_df(df: pd.DataFrame, feat_start=2, feat_end=2+FEAT_TOTAL):
    feats = df.iloc[:, feat_start:feat_end].astype(np.float32).values
    y_raw = df["opening_gains"].astype(np.float32).values
    code = df.iloc[:, 0].astype(str).values
    date = df.iloc[:, 1].astype(str).values
    return code, date, feats, y_raw


# ================== 数据集 ==================
class StockDataset(Dataset):
    def __init__(self, X, y_clip, y_raw, codes, dates):
        self.X = torch.from_numpy(X.astype(np.float32))
        self.y_clip = torch.from_numpy(y_clip.astype(np.float32))
        self.y_raw = torch.from_numpy(y_raw.astype(np.float32))
        self.codes = codes
        self.dates = dates

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            self.X[idx],
            self.y_clip[idx],
            self.y_raw[idx],
            self.codes[idx],
            self.dates[idx]
        )


# ================== MLP 模型 ==================
class MLPRegressor(nn.Module):
    def __init__(self, input_dim, hidden_sizes, dropout=0.0):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


# ================== 评估函数 ==================
def evaluate_daily_precision(scores: np.ndarray, labels: np.ndarray, dates: list,
                             top_list=EVAL_TOP_LIST, thr=0):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    metrics = {k: {"prec": [], "mean_y": []} for k in top_list}
    cover_days = 0

    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        s_day = scores[idxs]
        y_day = labels[idxs]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        cover_days += 1
        for k in top_list:
            k_eff = min(k, s_day.size)
            topk = order[:k_eff]
            yk = y_day[topk]
            hit = (yk >= thr).astype(np.float32)
            metrics[k]["prec"].append(hit.mean() if k_eff > 0 else 0.0)
            metrics[k]["mean_y"].append(yk.mean() if k_eff > 0 else 0.0)

    out = {}
    for k in top_list:
        out[k] = {
            "Precision@K": float(np.mean(metrics[k]["prec"])) if metrics[k]["prec"] else 0.0,
            "AvgReturn@K": float(np.mean(metrics[k]["mean_y"])) if metrics[k]["mean_y"] else 0.0,
            "CoveredDays": int(cover_days),
        }
    return out

def evaluate_daily_ndcg(scores: np.ndarray, labels: np.ndarray, dates: list,
                        top_list=EVAL_TOP_LIST, thr=OPEN_THR):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    def dcg_at_k(rels_sorted_by_rank, k):
        k_eff = min(k, len(rels_sorted_by_rank))
        if k_eff <= 0:
            return 0.0
        gains = (2 ** rels_sorted_by_rank[:k_eff] - 1.0)
        discounts = 1.0 / np.log2(np.arange(2, 2 + k_eff))
        return float(np.sum(gains * discounts))

    bucket = {k: [] for k in top_list}

    for d, idxs in idx_by_day.items():
        arr = np.array(idxs, dtype=np.int64)
        s_day = scores[arr]
        y_day = labels[arr]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        rel = (y_day >= thr).astype(np.int32)
        rel_sorted_gt = np.sort(rel)[::-1]

        for k in top_list:
            k_eff = min(k, s_day.size)
            if k_eff <= 0:
                bucket[k].append(0.0)
                continue
            rel_ranked = rel[order]
            dcg = dcg_at_k(rel_ranked, k_eff)
            idcg = dcg_at_k(rel_sorted_gt, k_eff)
            ndcg = (dcg / idcg) if idcg > 0 else 0.0
            bucket[k].append(ndcg)

    out = {}
    for k in top_list:
        vals = bucket[k]
        out[k] = {
            "NDCG@K": float(np.mean(vals)) if vals else 0.0,
            "CoveredDays": len(vals),
        }
    return out

def export_daily_reco(scores: np.ndarray, labels_raw: np.ndarray, codes: list, dates: list,
                      N=N_TARGET, path_csv="mlp_recommendations.csv"):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    rows = []
    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        order = np.argsort(-scores[idxs])
        topn = order[:min(N, idxs.size)]
        for r, j in enumerate(topn, start=1):
            i = idxs[j]
            rows.append({
                "date": dates[i],
                "rank": r,
                "code": codes[i],
                "score": float(scores[i]),
                "true_open_return": float(labels_raw[i])
            })
    pd.DataFrame(rows).sort_values(["date", "rank"]).to_csv(path_csv, index=False)
    print(f"[Export] Saved MLP recommendations to {path_csv}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)
    torch.backends.cuda.matmul.allow_tf32 = False

# ================== 主流程 ==================
def main():
    set_seed(42) 
    # --- 加载数据 ---
    train_df = pd.read_csv(TRAIN_CSV, low_memory=False)
    val_df   = pd.read_csv(VAL_CSV,   low_memory=False)
    test_df  = pd.read_csv(TEST_CSV,  low_memory=False)

    # 标签裁剪
    p1  = train_df["opening_gains"].quantile(0.01)
    p99 = train_df["opening_gains"].quantile(0.99)
    train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
    val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
    test_df["opening_gains_clipped"]  = test_df["opening_gains"].clip(lower=p1, upper=p99)

    tr_code, tr_date, tr_feats, tr_y_raw = cast_df(train_df)
    va_code, va_date, va_feats, va_y_raw = cast_df(val_df)
    te_code, te_date, te_feats, te_y_raw = cast_df(test_df)

    tr_y = train_df["opening_gains_clipped"].values.astype(np.float32)
    va_y = val_df["opening_gains_clipped"].values.astype(np.float32)
    te_y = test_df["opening_gains_clipped"].values.astype(np.float32)

    # --- 加载文本嵌入 ---
    tr_text = load_embeddings("train", tr_feats.shape[0])
    va_text = load_embeddings("val",   va_feats.shape[0])
    te_text = load_embeddings("test",  te_feats.shape[0])

    # --- 拼接特征 ---
    X_train = np.concatenate([tr_feats, tr_text], axis=1)
    X_val   = np.concatenate([va_feats, va_text], axis=1)
    X_test  = np.concatenate([te_feats, te_text], axis=1)

    print(f"Final feature dim: {X_train.shape[1]}")

    # --- 特征标准化（对 MLP 至关重要）---
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)

    #--- 构建 DataLoader ---
    train_dataset = StockDataset(X_train_scaled, tr_y, tr_y_raw, tr_code, tr_date)
    val_dataset   = StockDataset(X_val_scaled,   va_y, va_y_raw, va_code, va_date)
    test_dataset  = StockDataset(X_test_scaled,  te_y, te_y_raw, te_code, te_date)

    def seed_worker(worker_id):
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)
        torch.manual_seed(worker_seed)

    g = torch.Generator()
    g.manual_seed(42)

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
        num_workers=4, pin_memory=True,
        worker_init_fn=seed_worker,
        generator=g,
        persistent_workers=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
        num_workers=4, pin_memory=True,
        worker_init_fn=seed_worker,
        generator=g,
        persistent_workers=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False, 
        num_workers=4, pin_memory=True,
        worker_init_fn=seed_worker,
        generator=g,
        persistent_workers=True
    )

    # --- 初始化模型 ---
    input_dim = X_train_scaled.shape[1]
    model = MLPRegressor(input_dim, HIDDEN_SIZES, dropout=DROP_RATE).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.MSELoss()

    # --- 训练循环 ---
    best_val_loss = float('inf')
    patience_counter = 0
    best_path = "results/08_mlp.pth"

    print(f"\n[Training MLP on {DEVICE}]")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            X, y_clip, _, _, _ = [x.to(DEVICE, non_blocking=True) if isinstance(x, torch.Tensor) else x for x in batch]
            optimizer.zero_grad()
            pred = model(X)
            loss = criterion(pred, y_clip)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # 验证
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                X, y_clip, _, _, _ = [x.to(DEVICE, non_blocking=True) if isinstance(x, torch.Tensor) else x for x in batch]
                pred = model(X)
                val_loss += criterion(pred, y_clip).item()

        tr_loss = total_loss / len(train_loader)
        val_loss = val_loss / len(val_loader)

        print(f"[Epoch {epoch}] Train Loss: {tr_loss:.6f} | Val Loss: {val_loss:.6f}")

        # 早停
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'scaler': scaler,
                'input_dim': input_dim,
                'hidden_sizes': HIDDEN_SIZES
            }, best_path)
            print(f"  >> Saved best model to {best_path}")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

    # --- 测试评估 ---
    checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model = MLPRegressor(input_dim, HIDDEN_SIZES, dropout=DROP_RATE).to(DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    all_scores, all_labels, all_codes, all_dates = [], [], [], []
    with torch.no_grad():
        for batch in test_loader:
            X, _, y_raw, codes, dates = batch
            X = X.to(DEVICE, non_blocking=True)
            scores = model(X).cpu().numpy()
            all_scores.append(scores)
            all_labels.append(y_raw.numpy())
            all_codes.extend(codes)
            all_dates.extend(dates)

    pred_test = np.concatenate(all_scores, axis=0)
    true_test = np.concatenate(all_labels, axis=0)

    # --- 评估 ---
    print("\n[Evaluation on TEST set]")
    test_metrics = evaluate_daily_precision(pred_test, true_test, all_dates, top_list=EVAL_TOP_LIST, thr=OPEN_THR)
    test_ndcg    = evaluate_daily_ndcg(pred_test, true_test, all_dates, top_list=EVAL_TOP_LIST, thr=OPEN_THR)

    for k in EVAL_TOP_LIST:
        print(f"MLP TEST P@{k}={test_metrics[k]['Precision@K']:.4f} | "
              f"Avg@{k}={test_metrics[k]['AvgReturn@K']:.4f} | "
              f"NDCG@{k}={test_ndcg[k]['NDCG@K']:.4f} | "
              f"Days={test_metrics[k]['CoveredDays']}")

    # --- 导出推荐 ---
    export_daily_reco(pred_test, true_test, all_codes, all_dates, N=N_TARGET, path_csv="results/08_mlp.csv")


if __name__ == "__main__":
    main()

/tmp/ipykernel_9733/2681421544.py:232: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["opening_gains_clipped"] = train_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_9733/2681421544.py:233: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val_df["opening_gains_clipped"]   = val_df["opening_gains"].clip(lower=p1, upper=p99)
/tmp/ipykernel_9733/2681421544.py:234: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Cons

[Info] Loaded train embeddings: shape=(1610196, 1024)
[Info] Loaded val embeddings: shape=(349618, 1024)
[Info] Loaded test embeddings: shape=(604975, 1024)
Final feature dim: 1329

[Training MLP on cuda]
[Epoch 1] Train Loss: 0.646115 | Val Loss: 0.881190
  >> Saved best model to results/08_mlp.pth
[Epoch 2] Train Loss: 0.612729 | Val Loss: 0.865351
  >> Saved best model to results/08_mlp.pth
[Epoch 3] Train Loss: 0.602026 | Val Loss: 0.876371
[Epoch 4] Train Loss: 0.595994 | Val Loss: 0.870977
[Epoch 5] Train Loss: 0.591834 | Val Loss: 0.869235
[Epoch 6] Train Loss: 0.588045 | Val Loss: 0.869260
[Epoch 7] Train Loss: 0.585421 | Val Loss: 0.897113
[Epoch 8] Train Loss: 0.582517 | Val Loss: 0.871133
[Epoch 9] Train Loss: 0.579601 | Val Loss: 0.872510
[Epoch 10] Train Loss: 0.577702 | Val Loss: 0.874534
[Epoch 11] Train Loss: 0.575573 | Val Loss: 0.875909
[Epoch 12] Train Loss: 0.573464 | Val Loss: 0.883451
[Epoch 13] Train Loss: 0.571864 | Val Loss: 0.876138
[Epoch 14] Train Loss: 0.57

# Logistic Regression

In [3]:
###### logistic_regression_baseline.py
import os
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import List

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ================== 路径与参数 ==================
TRAIN_CSV = "/root/autodl-tmp/train_filtered_filtered.csv"
VAL_CSV   = "/root/autodl-tmp/val_filtered_filtered.csv"
TEST_CSV  = "/root/autodl-tmp/test_filtered_filtered.csv"

TELEGRAM_EMB_DIR = "/root/autodl-tmp/telegram_emb_cache_clean"

# 特征维度（必须与主模型一致）
DAILY_T, DAILY_F = 5, 43
MIN_T, MIN_F     = 5, 18
FEAT_TOTAL       = DAILY_T * DAILY_F + MIN_T * MIN_F

# 业务目标（用于构造二分类标签）
OPEN_THR = 0.0
N_TARGET = 20
EVAL_TOP_LIST = [5, 10, 20]

# Logistic Regression 超参
LOGREG_C = 1.0  # 正则强度（越小正则越强）
MAX_ITER = 1000


# ================== 工具函数 ==================
def load_embeddings(split: str, expected_n: int) -> np.ndarray:
    path = os.path.join(TELEGRAM_EMB_DIR, f"{split}_telegram_emb_filtered_filtered.npy")
    if not os.path.exists(path):
        raise FileNotFoundError(f"Embedding file not found: {path}")
    emb = np.load(path).astype(np.float32)
    assert emb.shape[0] == expected_n, f"{split} emb rows mismatch"
    print(f"[Info] Loaded {split} embeddings: shape={emb.shape}")
    return emb

def cast_df(df: pd.DataFrame, feat_start=2, feat_end=2+FEAT_TOTAL):
    feats = df.iloc[:, feat_start:feat_end].astype(np.float32).values
    y_raw = df["opening_gains"].astype(np.float32).values
    code = df.iloc[:, 0].astype(str).values
    date = df.iloc[:, 1].astype(str).values
    return code, date, feats, y_raw


# ================== 评估函数（复用逻辑） ==================
def evaluate_daily_precision(scores: np.ndarray, labels: np.ndarray, dates: list,
                             top_list=EVAL_TOP_LIST, thr=0):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    metrics = {k: {"prec": [], "mean_y": []} for k in top_list}
    cover_days = 0

    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        s_day = scores[idxs]
        y_day = labels[idxs]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        cover_days += 1
        for k in top_list:
            k_eff = min(k, s_day.size)
            topk = order[:k_eff]
            yk = y_day[topk]
            hit = (yk >= thr).astype(np.float32)
            metrics[k]["prec"].append(hit.mean() if k_eff > 0 else 0.0)
            metrics[k]["mean_y"].append(yk.mean() if k_eff > 0 else 0.0)

    out = {}
    for k in top_list:
        out[k] = {
            "Precision@K": float(np.mean(metrics[k]["prec"])) if metrics[k]["prec"] else 0.0,
            "AvgReturn@K": float(np.mean(metrics[k]["mean_y"])) if metrics[k]["mean_y"] else 0.0,
            "CoveredDays": int(cover_days),
        }
    return out

def evaluate_daily_ndcg(scores: np.ndarray, labels: np.ndarray, dates: list,
                        top_list=EVAL_TOP_LIST, thr=OPEN_THR):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    def dcg_at_k(rels_sorted_by_rank, k):
        k_eff = min(k, len(rels_sorted_by_rank))
        if k_eff <= 0:
            return 0.0
        gains = (2 ** rels_sorted_by_rank[:k_eff] - 1.0)
        discounts = 1.0 / np.log2(np.arange(2, 2 + k_eff))
        return float(np.sum(gains * discounts))

    bucket = {k: [] for k in top_list}

    for d, idxs in idx_by_day.items():
        arr = np.array(idxs, dtype=np.int64)
        s_day = scores[arr]
        y_day = labels[arr]
        if s_day.size == 0:
            continue
        order = np.argsort(-s_day)
        rel = (y_day >= thr).astype(np.int32)
        rel_sorted_gt = np.sort(rel)[::-1]

        for k in top_list:
            k_eff = min(k, s_day.size)
            if k_eff <= 0:
                bucket[k].append(0.0)
                continue
            rel_ranked = rel[order]
            dcg = dcg_at_k(rel_ranked, k_eff)
            idcg = dcg_at_k(rel_sorted_gt, k_eff)
            ndcg = (dcg / idcg) if idcg > 0 else 0.0
            bucket[k].append(ndcg)

    out = {}
    for k in top_list:
        vals = bucket[k]
        out[k] = {
            "NDCG@K": float(np.mean(vals)) if vals else 0.0,
            "CoveredDays": len(vals),
        }
    return out

def export_daily_reco(scores: np.ndarray, labels_raw: np.ndarray, codes: list, dates: list,
                      N=N_TARGET, path_csv="logreg_recommendations.csv"):
    idx_by_day = defaultdict(list)
    for i, d in enumerate(dates):
        idx_by_day[d].append(i)

    rows = []
    for d, idxs in idx_by_day.items():
        idxs = np.array(idxs, dtype=np.int64)
        order = np.argsort(-scores[idxs])
        topn = order[:min(N, idxs.size)]
        for r, j in enumerate(topn, start=1):
            i = idxs[j]
            rows.append({
                "date": dates[i],
                "rank": r,
                "code": codes[i],
                "score": float(scores[i]),
                "true_open_return": float(labels_raw[i])
            })
    pd.DataFrame(rows).sort_values(["date", "rank"]).to_csv(path_csv, index=False)
    print(f"[Export] Saved Logistic Regression recommendations to {path_csv}")


# ================== 主流程 ==================
def main():
    # --- 加载数据 ---
    train_df = pd.read_csv(TRAIN_CSV, low_memory=False)
    val_df   = pd.read_csv(VAL_CSV,   low_memory=False)
    test_df  = pd.read_csv(TEST_CSV,  low_memory=False)

    tr_code, tr_date, tr_feats, tr_y_raw = cast_df(train_df)
    va_code, va_date, va_feats, va_y_raw = cast_df(val_df)
    te_code, te_date, te_feats, te_y_raw = cast_df(test_df)

    # 构造二分类标签：是否高开 ≥ OPEN_THR
    tr_y_binary = (tr_y_raw >= OPEN_THR).astype(int)
    va_y_binary = (va_y_raw >= OPEN_THR).astype(int)
    te_y_binary = (te_y_raw >= OPEN_THR).astype(int)

    # --- 加载文本嵌入 ---
    tr_text = load_embeddings("train", tr_feats.shape[0])
    va_text = load_embeddings("val",   va_feats.shape[0])
    te_text = load_embeddings("test",  te_feats.shape[0])

    # --- 拼接特征 ---
    X_train = np.concatenate([tr_feats, tr_text], axis=1)
    X_val   = np.concatenate([va_feats, va_text], axis=1)
    X_test  = np.concatenate([te_feats, te_text], axis=1)

    print(f"Final feature dim: {X_train.shape[1]}")

    # --- 特征标准化（对 Logistic Regression 至关重要）---
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled   = scaler.transform(X_val)
    X_test_scaled  = scaler.transform(X_test)

    # --- 训练 Logistic Regression ---
    print("\n[Training Logistic Regression ...]")
    model = LogisticRegression(
        C=LOGREG_C,
        penalty='l2',
        solver='lbfgs',  # 适合中小规模
        max_iter=MAX_ITER,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_scaled, tr_y_binary)

    # --- 预测：输出 positive class 的概率作为 score ---
    pred_val_proba = model.predict_proba(X_val_scaled)[:, 1]   # P(y=1)
    pred_test_proba = model.predict_proba(X_test_scaled)[:, 1]

    # --- 评估（使用原始连续标签计算 AvgReturn@K）---
    print("\n[Evaluation on TEST set]")
    test_metrics = evaluate_daily_precision(pred_test_proba, te_y_raw, te_date, top_list=EVAL_TOP_LIST, thr=OPEN_THR)
    test_ndcg    = evaluate_daily_ndcg(pred_test_proba, te_y_raw, te_date, top_list=EVAL_TOP_LIST, thr=OPEN_THR)

    for k in EVAL_TOP_LIST:
        print(f"LogReg TEST P@{k}={test_metrics[k]['Precision@K']:.4f} | "
              f"Avg@{k}={test_metrics[k]['AvgReturn@K']:.4f} | "
              f"NDCG@{k}={test_ndcg[k]['NDCG@K']:.4f} | "
              f"Days={test_metrics[k]['CoveredDays']}")

    # --- 导出推荐 ---
    export_daily_reco(pred_test_proba, te_y_raw, te_code, te_date, N=N_TARGET, path_csv="results/09_logistic_regression.csv")

    # --- 可选：保存系数 ---
    coef = model.coef_[0]
    np.save("results/09_logistic_regression.npy", coef)
    print(f"[Info] Model coefficients saved to logreg_coef.npy (shape={coef.shape})")


if __name__ == "__main__":
    main()

[Info] Loaded train embeddings: shape=(1610196, 1024)
[Info] Loaded val embeddings: shape=(349618, 1024)
[Info] Loaded test embeddings: shape=(604975, 1024)
Final feature dim: 1329

[Training Logistic Regression ...]


/root/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/root/miniconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



[Evaluation on TEST set]
LogReg TEST P@5=0.6244 | Avg@5=-0.0778 | NDCG@5=0.6212 | Days=205
LogReg TEST P@10=0.6195 | Avg@10=-0.0804 | NDCG@10=0.6189 | Days=205
LogReg TEST P@20=0.6100 | Avg@20=-0.0901 | NDCG@20=0.6124 | Days=205
[Export] Saved Logistic Regression recommendations to results/09_logistic_regression.csv
[Info] Model coefficients saved to logreg_coef.npy (shape=(1329,))


# SGD Linear Regression

In [4]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from typing import Dict, List

from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

# ================== 路径 ==================
TRAIN_CSV = "/root/autodl-tmp/train_filtered_filtered.csv"
VAL_CSV   = "/root/autodl-tmp/val_filtered_filtered.csv"
TEST_CSV  = "/root/autodl-tmp/test_filtered_filtered.csv"

TELEGRAM_EMB_DIR = "/root/autodl-tmp/telegram_emb_cache_clean"

# ================== 特征维度 ==================
DAILY_T, DAILY_F = 5, 43
MIN_T, MIN_F     = 5, 18
FEAT_TOTAL       = DAILY_T * DAILY_F + MIN_T * MIN_F

# ================== 任务设置 ==================
OPEN_THR = 0.0
N_TARGET = 20
EVAL_TOP_LIST = [5, 10, 20]

# ================== 超参数搜索 ==================
ALPHA_LIST = [1e-3,1e-4,1e-5]

# ================== 工具函数 ==================

def load_embeddings(split, n):

    path = os.path.join(TELEGRAM_EMB_DIR,f"{split}_telegram_emb_filtered_filtered.npy")

    emb = np.load(path).astype(np.float32)

    assert emb.shape[0] == n

    return emb


def cast_df(df):

    feats = df.iloc[:,2:2+FEAT_TOTAL].astype(np.float32).values

    y = df["opening_gains"].astype(np.float32).values

    code = df.iloc[:,0].astype(str).values

    date = df.iloc[:,1].astype(str).values

    return code,date,feats,y


# ================== Precision@K ==================

def evaluate_daily_precision(scores,labels,dates,top_list,thr):

    idx_by_day = defaultdict(list)

    for i,d in enumerate(dates):
        idx_by_day[d].append(i)

    metrics = {k:[] for k in top_list}

    for d,idxs in idx_by_day.items():

        idxs = np.array(idxs)

        s_day = scores[idxs]

        y_day = labels[idxs]

        order = np.argsort(-s_day)

        for k in top_list:

            k_eff = min(k,len(order))

            topk = order[:k_eff]

            hit = (y_day[topk] >= thr).astype(float)

            metrics[k].append(hit.mean())

    return {k:np.mean(v) for k,v in metrics.items()}


# ================== NDCG ==================

def evaluate_daily_ndcg(scores,labels,dates,top_list,thr):

    idx_by_day = defaultdict(list)

    for i,d in enumerate(dates):
        idx_by_day[d].append(i)

    def dcg(rel):

        return np.sum((2**rel-1)/np.log2(np.arange(2,len(rel)+2)))

    metrics = {k:[] for k in top_list}

    for d,idxs in idx_by_day.items():

        idxs = np.array(idxs)

        s = scores[idxs]

        y = labels[idxs]

        rel = (y>=thr).astype(int)

        order = np.argsort(-s)

        rel_sorted = np.sort(rel)[::-1]

        for k in top_list:

            r = rel[order][:k]

            ideal = rel_sorted[:k]

            idcg = dcg(ideal)

            if idcg==0:
                metrics[k].append(0)
            else:
                metrics[k].append(dcg(r)/idcg)

    return {k:np.mean(v) for k,v in metrics.items()}


# ================== AvgReturn ==================

def evaluate_avg_return(scores,labels,dates,top_list):

    idx_by_day = defaultdict(list)

    for i,d in enumerate(dates):
        idx_by_day[d].append(i)

    metrics = {k:[] for k in top_list}

    for d,idxs in idx_by_day.items():

        idxs = np.array(idxs)

        s = scores[idxs]

        y = labels[idxs]

        order = np.argsort(-s)

        for k in top_list:

            k_eff = min(k,len(order))

            topk = order[:k_eff]

            metrics[k].append(y[topk].mean())

    return {k:np.mean(v) for k,v in metrics.items()}


# ================== 导出推荐 ==================

def export_daily_reco(scores,labels,codes,dates,N):

    idx_by_day = defaultdict(list)

    for i,d in enumerate(dates):
        idx_by_day[d].append(i)

    rows=[]

    for d,idxs in idx_by_day.items():

        idxs=np.array(idxs)

        order=np.argsort(-scores[idxs])

        top=order[:min(N,len(order))]

        for r,j in enumerate(top,start=1):

            i=idxs[j]

            rows.append({
                "date":dates[i],
                "rank":r,
                "code":codes[i],
                "score":float(scores[i]),
                "true_open_return":float(labels[i])
            })

    pd.DataFrame(rows).sort_values(["date","rank"]).to_csv("results/10_sgd_linear_regression.csv",index=False)


# ================== 主程序 ==================

def main():

    print("Loading data...")

    train_df=pd.read_csv(TRAIN_CSV)
    val_df=pd.read_csv(VAL_CSV)
    test_df=pd.read_csv(TEST_CSV)

    tr_code,tr_date,tr_feats,tr_y=cast_df(train_df)
    va_code,va_date,va_feats,va_y=cast_df(val_df)
    te_code,te_date,te_feats,te_y=cast_df(test_df)

    tr_text=load_embeddings("train",len(tr_feats))
    va_text=load_embeddings("val",len(va_feats))
    te_text=load_embeddings("test",len(te_feats))

    X_train=np.concatenate([tr_feats,tr_text],axis=1)
    X_val=np.concatenate([va_feats,va_text],axis=1)
    X_test=np.concatenate([te_feats,te_text],axis=1)

    scaler=StandardScaler()

    X_train=scaler.fit_transform(X_train)
    X_val=scaler.transform(X_val)
    X_test=scaler.transform(X_test)

    print("Feature dim:",X_train.shape[1])

    best_alpha = None
    best_score = -np.inf

    print("\n===== Hyperparameter Search =====")

    for alpha in ALPHA_LIST:

        model = SGDRegressor(
            loss="squared_error",
            penalty="l2",
            alpha=alpha,
            max_iter=2000,
            tol=1e-4,
            random_state=42
        )

        model.fit(X_train, tr_y)

        pred_val = model.predict(X_val)

        val_avg5 = evaluate_avg_return(pred_val, va_y, va_date, [5])[5]

        print(f"alpha={alpha}  Val AvgReturn@5={val_avg5:.4f}")

        if val_avg5 > best_score:
            best_score = val_avg5
            best_alpha = alpha

    print("\nBest alpha:",best_alpha)

    # ================== 训练最终模型 ==================

    final_model=SGDRegressor(
        loss="squared_error",
        penalty="l2",
        alpha=best_alpha,
        max_iter=2000,
        tol=1e-4,
        random_state=42
    )

    final_model.fit(X_train,tr_y)

    # ================== Test Evaluation ==================

    pred_test=final_model.predict(X_test)

    precision=evaluate_daily_precision(pred_test,te_y,te_date,EVAL_TOP_LIST,OPEN_THR)

    ndcg=evaluate_daily_ndcg(pred_test,te_y,te_date,EVAL_TOP_LIST,OPEN_THR)

    avg_return=evaluate_avg_return(pred_test,te_y,te_date,EVAL_TOP_LIST)

    print("\n===== TEST RESULT =====")

    for k in EVAL_TOP_LIST:

        print(
            f"P@{k}={precision[k]:.4f} | "
            f"NDCG@{k}={ndcg[k]:.4f} | "
            f"Return@{k}={avg_return[k]:.4f}"
        )

    # ================== 导出推荐 ==================

    export_daily_reco(pred_test,te_y,te_code,te_date,N_TARGET)

    print("\nRecommendations saved.")


if __name__=="__main__":
    main()

Loading data...
Feature dim: 1329

===== Hyperparameter Search =====
alpha=0.001  Val AvgReturn@5=-0.0420
alpha=0.0001  Val AvgReturn@5=-0.1970
alpha=1e-05  Val AvgReturn@5=-0.3937

Best alpha: 0.001

===== TEST RESULT =====
P@5=0.5473 | NDCG@5=0.5527 | Return@5=-0.0678
P@10=0.5293 | NDCG@10=0.5388 | Return@10=-0.1046
P@20=0.5249 | NDCG@20=0.5325 | Return@20=-0.1501

Recommendations saved.
